MIT-BIH Arrhythmia Classification — Phase 1: Patient-Grouped Evaluation
Purpose: establish a fair, reproducible starting point before changing features or tuning models.

This notebook:
1. Loads the prepared heartbeat CSV and (if available) the Stage 7 checkpoint.
2. maps original MIT-BIH annotation symbols to five AAMI-style classes (N, S, V, F, Q) and excludes the documented non-beat/unsupported symbols.
3. audits old record-grouped folds, including records 201 and 202.
4. maps each recording to a person ID, grouping 201 and 202 together.
5. creates five repeated, patient-grouped 5-fold split sets and saves them.
6. evaluates P1, P2, and P3 on identical saved splits, recording predictions, class probabilities, per-class metrics, confusion matrices, accuracy, balanced accuracy, and Macro-F1.
Important: P1 uses 100 trees as specified. P2 and P3 use the same fixed Random Forest settings (200 trees; no search). Feature selection and oversampling are fitted/applied on training data only. The validation fold is never oversampled.



In [1]:
# 1. Imports and reproducibility
import os
from pathlib import Path
import pickle
import json
import warnings

import numpy as np
import pandas as pd
import sklearn
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, f1_score,
    precision_recall_fscore_support, confusion_matrix
)

from IPython.display import display

SEED = 42
CLASS_ORDER = ["N", "S", "V", "F", "Q"]
N_REPEATS = 5
N_SPLITS = 5

print("pandas:", pd.__version__)
print("scikit-learn:", sklearn.__version__)


pandas: 2.3.3
scikit-learn: 1.6.1


In [2]:
# 2. Locate input files in Kaggle
# Kaggle datasets are normally mounted somewhere under /kaggle/input.
INPUT_ROOT = Path("/kaggle/input")

def find_input_file(filename):
    matches = list(INPUT_ROOT.rglob(filename)) if INPUT_ROOT.exists() else []
    if not matches:
        # Also allow files uploaded directly to the current working directory.
        local = Path(filename)
        if local.exists():
            return local
        raise FileNotFoundError(
            f"Could not find {filename}. Add it as a Kaggle input or edit this cell."
        )
    if len(matches) > 1:
        print(f"Found multiple copies of {filename}; using: {matches[0]}")
    return matches[0]

DATA_PATH = find_input_file("MITBIH_MLII_DualLead_Dataset.csv")
print("Dataset:", DATA_PATH)

try:
    CHECKPOINT_PATH = find_input_file("mit_bih_stage7_final_checkpoint.pkl")
    print("Stage 7 checkpoint:", CHECKPOINT_PATH)
except FileNotFoundError:
    CHECKPOINT_PATH = None
    print("Stage 7 checkpoint not found. Derivative features will be recomputed from the CSV.")


Dataset: /kaggle/input/datasets/aakashshah3799/mit-bih-own/MITBIH_MLII_DualLead_Dataset.csv
Stage 7 checkpoint: /kaggle/input/datasets/aakashshah3799/mit-bih-own/mit_bih_stage7_final_checkpoint.pkl


In [5]:
# 3. Load and validate the prepared dataset
raw_df = pd.read_csv(DATA_PATH)
print("Raw shape:", raw_df.shape)
print("Columns (first 8):", raw_df.columns[:8].tolist())
print("Columns (last 8):", raw_df.columns[-8:].tolist())
print("Unique recordings:", raw_df["Record_ID"].nunique())
print("Original annotation counts:")
display(raw_df["Label"].value_counts(dropna=False).rename_axis("Original_Label").to_frame("Count"))

# Documented AAMI-style mapping used in the previous project pipeline.
AAMI_MAP = {
    "N": "N", "L": "N", "R": "N", "e": "N", "j": "N",
    "A": "S", "a": "S", "J": "S", "S": "S",
    "V": "V", "E": "V",
    "F": "F",
    "/": "Q", "f": "Q", "Q": "Q"
}

aami_df = raw_df.loc[raw_df["Label"].isin(AAMI_MAP)].copy().reset_index(drop=True)
aami_df["AAMI_Label"] = aami_df["Label"].map(AAMI_MAP)

signal_cols = [c for c in aami_df.columns if c.startswith("MLII_") or c.startswith("Lead2_")]
mlii_cols = [f"MLII_{i}" for i in range(201)]
lead2_cols = [f"Lead2_{i}" for i in range(201)]

missing_signal_cols = sorted(set(mlii_cols + lead2_cols) - set(aami_df.columns))
if missing_signal_cols:
    raise ValueError(f"Expected signal columns missing, e.g. {missing_signal_cols[:10]}")

print("AAMI dataset shape:", aami_df.shape)
print("Rows excluded by mapping:", len(raw_df) - len(aami_df))
print("AAMI class counts:")
display(aami_df["AAMI_Label"].value_counts().reindex(CLASS_ORDER).rename_axis("AAMI_Class").to_frame("Count"))
print("Missing values in AAMI data:", int(aami_df.isna().sum().sum()))


Raw shape: (106870, 404)
Columns (first 8): ['Record_ID', 'MLII_0', 'MLII_1', 'MLII_2', 'MLII_3', 'MLII_4', 'MLII_5', 'MLII_6']
Columns (last 8): ['Lead2_194', 'Lead2_195', 'Lead2_196', 'Lead2_197', 'Lead2_198', 'Lead2_199', 'Lead2_200', 'Label']
Unique recordings: 46
Original annotation counts:


,Count
Original_Label,
N,74766
L,8073
R,7257
V,7123
/,3619
A,2546
F,802
~,577
!,472


AAMI dataset shape: (105047, 405)
Rows excluded by mapping: 1823
AAMI class counts:


,Count
AAMI_Class,
N,90341
S,2781
V,7229
F,802
Q,3894


Missing values in AAMI data: 0


In [6]:
# 4. Map Record_ID to Person_ID
# MIT-BIH records 201 and 202 are recordings from the same person.
# Other records are assigned distinct IDs unless a verified subject mapping is added.
aami_df["Person_ID"] = aami_df["Record_ID"].astype(str)
aami_df.loc[aami_df["Record_ID"].isin([201, 202]), "Person_ID"] = "person_201_202"

record_person = (
    aami_df[["Record_ID", "Person_ID"]]
    .drop_duplicates()
    .sort_values("Record_ID")
    .reset_index(drop=True)
)
assert record_person["Record_ID"].is_unique
print("Record-to-person mapping rows:", len(record_person))
print("Unique Person_ID values:", record_person["Person_ID"].nunique())
display(record_person[record_person["Record_ID"].isin([201, 202])])


Record-to-person mapping rows: 46
Unique Person_ID values: 45


,Record_ID,Person_ID
22,201,person_201_202
23,202,person_201_202


In [7]:
# 5. Record-by-class counts
record_class_counts = pd.crosstab(aami_df["Record_ID"], aami_df["AAMI_Label"])
record_class_counts = record_class_counts.reindex(columns=CLASS_ORDER, fill_value=0).sort_index()
display(record_class_counts)
record_class_counts.to_csv("record_by_AAMI_class_counts.csv")
record_person.to_csv("record_to_person_mapping.csv", index=False)
print("Saved: record_by_AAMI_class_counts.csv, record_to_person_mapping.csv")


AAMI_Label,N,S,V,F,Q
Record_ID,,,,,
100,2237,33,1,0,0
101,1859,3,0,0,2
103,2082,2,0,0,0
105,2526,0,41,0,5
106,1507,0,520,0,0
107,0,0,59,0,2077
108,1739,4,17,2,0
109,2491,0,38,2,0
111,2123,0,1,0,0


Saved: record_by_AAMI_class_counts.csv, record_to_person_mapping.csv


In [8]:
# 6. Audit the old Stage 7 folds (if checkpoint is available)
old_fold_audit = None
checkpoint = None

if CHECKPOINT_PATH is not None:
    with open(CHECKPOINT_PATH, "rb") as f:
        checkpoint = pickle.load(f)

    print("Checkpoint name:", checkpoint.get("checkpoint_name"))
    print("Created at:", checkpoint.get("created_at"))
    print("Saved best result:", checkpoint.get("best_result"))

    old_data = checkpoint.get("data", {})
    old_y = old_data.get("y")
    old_groups = old_data.get("groups")
    old_fold_data = old_data.get("fold_data", {})

    if old_y is not None and old_groups is not None:
        # Validate that checkpoint rows align with the current AAMI dataset.
        labels_match = list(pd.Series(old_y).astype(str).values) == list(aami_df["AAMI_Label"].astype(str).values)
        groups_match = list(pd.Series(old_groups).astype(int).values) == list(aami_df["Record_ID"].astype(int).values)
        print("Checkpoint labels align with CSV after mapping:", labels_match)
        print("Checkpoint Record_IDs align with CSV after mapping:", groups_match)
        if not (labels_match and groups_match):
            raise ValueError(
                "Checkpoint rows do not align with the current CSV after AAMI mapping. "
                "Do not reuse checkpoint fold indices until this mismatch is resolved."
            )

        fold_rows = []
        old_val_fold_by_row = {}
        for fold_name, split in old_fold_data.items():
            for idx in np.asarray(split["val_idx"], dtype=int):
                old_val_fold_by_row[int(idx)] = fold_name
            fold_rows.append({
                "Fold": fold_name,
                "Train_Rows": len(split["train_idx"]),
                "Validation_Rows": len(split["val_idx"])
            })
        old_fold_audit = pd.DataFrame(fold_rows)
        display(old_fold_audit)

        for rid in [201, 202]:
            rid_rows = np.flatnonzero(aami_df["Record_ID"].to_numpy() == rid)
            rid_folds = sorted({old_val_fold_by_row.get(int(i), "NOT_FOUND") for i in rid_rows})
            print(f"Record {rid}: old validation fold(s) = {rid_folds}; rows = {len(rid_rows)}")

        # Verify no old validation fold contains both recordings as different records.
        fold201 = {old_val_fold_by_row.get(int(i)) for i in np.flatnonzero(aami_df["Record_ID"].to_numpy() == 201)}
        fold202 = {old_val_fold_by_row.get(int(i)) for i in np.flatnonzero(aami_df["Record_ID"].to_numpy() == 202)}
        print("201 and 202 assigned to the same old validation fold:", fold201 == fold202)
else:
    print("No checkpoint available; skipping old-fold audit.")


Checkpoint name: MIT-BIH Stage 7 Complete
Created at: 2026-08-29T06:59:53.199515
Saved best result: {'optimization_method': 'Randomized Search', 'accuracy': 0.870729, 'balanced_accuracy': 0.362149, 'macro_f1': 0.348888}
Checkpoint labels align with CSV after mapping: True
Checkpoint Record_IDs align with CSV after mapping: True


,Fold,Train_Rows,Validation_Rows
0,fold_1,80543,24504
1,fold_2,80984,24063
2,fold_3,86520,18527
3,fold_4,86508,18539
4,fold_5,85633,19414


Record 201: old validation fold(s) = ['fold_4']; rows = 1963
Record 202: old validation fold(s) = ['fold_5']; rows = 2136
201 and 202 assigned to the same old validation fold: False


In [9]:
# 7. Build the 16 derivative features
# If the checkpoint's derivative features align exactly, reuse them.
# Otherwise compute the same named features from the 201-sample beat windows.

def derivative_features_for_lead(values, prefix):
    d1 = np.diff(values, axis=1)
    d2 = np.diff(d1, axis=1)
    return {
        f"{prefix}_diff_mean": d1.mean(axis=1),
        f"{prefix}_diff_std": d1.std(axis=1),
        f"{prefix}_diff_max": d1.max(axis=1),
        f"{prefix}_diff_min": d1.min(axis=1),
        f"{prefix}_diff_abs_mean": np.abs(d1).mean(axis=1),
        f"{prefix}_diff_energy": np.sum(d1 ** 2, axis=1),
        f"{prefix}_second_diff_std": d2.std(axis=1),
        f"{prefix}_second_diff_abs_mean": np.abs(d2).mean(axis=1),
    }

def compute_derivative_features(frame):
    x1 = frame[mlii_cols].to_numpy(dtype=np.float64)
    x2 = frame[lead2_cols].to_numpy(dtype=np.float64)
    features = {}
    features.update(derivative_features_for_lead(x1, "MLII"))
    features.update(derivative_features_for_lead(x2, "Lead2"))
    return pd.DataFrame(features, index=frame.index)

DERIVATIVE_FEATURES = None
if checkpoint is not None:
    candidate = checkpoint.get("data", {}).get("derivative_features")
    if isinstance(candidate, pd.DataFrame) and candidate.shape == (len(aami_df), 16):
        DERIVATIVE_FEATURES = candidate.reset_index(drop=True).copy()
        print("Reusing 16 derivative features from the checkpoint.")
if DERIVATIVE_FEATURES is None:
    DERIVATIVE_FEATURES = compute_derivative_features(aami_df).reset_index(drop=True)
    print("Computed derivative features from the CSV.")

print("Derivative feature shape:", DERIVATIVE_FEATURES.shape)
display(DERIVATIVE_FEATURES.head())


Reusing 16 derivative features from the checkpoint.
Derivative feature shape: (105047, 16)


,MLII_diff_mean,MLII_diff_std,MLII_diff_max,MLII_diff_min,MLII_diff_abs_mean,MLII_diff_energy,MLII_second_diff_std,MLII_second_diff_abs_mean,Lead2_diff_mean,Lead2_diff_std,Lead2_diff_max,Lead2_diff_min,Lead2_diff_abs_mean,Lead2_diff_energy,Lead2_second_diff_std,Lead2_second_diff_abs_mean
0,-0.135,11.134486,51,-65,4.635,24799,5.264835,3.336683,-0.180,7.810096,25,-54,3.450,12206,4.703212,3.055276
1,-0.080,13.089064,58,-94,4.730,34266,7.261103,3.738693,-0.135,9.209602,30,-73,3.805,16967,6.122690,3.145729
2,0.000,12.299187,56,-82,4.680,30254,6.693939,3.552764,-0.080,8.917601,26,-67,3.820,15906,5.932194,3.190955
3,-0.095,11.774803,58,-78,4.535,27731,6.593355,3.683417,-0.130,8.048795,26,-65,3.550,12960,5.973131,3.477387
4,-0.105,11.993914,56,-78,4.585,28773,6.331287,3.532663,-0.075,8.573178,27,-71,3.595,14701,6.170474,3.261307


In [10]:
# 8. Create repeated patient-grouped 5-fold splits and save them
# Each repeat uses a different seed. All rows from one Person_ID stay together.
y = aami_df["AAMI_Label"].to_numpy()
record_ids = aami_df["Record_ID"].to_numpy()
person_ids = aami_df["Person_ID"].to_numpy()
row_ids = np.arange(len(aami_df))

split_indices = {}
split_rows = []

for repeat in range(1, N_REPEATS + 1):
    splitter = StratifiedGroupKFold(
        n_splits=N_SPLITS,
        shuffle=True,
        random_state=SEED + repeat - 1
    )
    repeat_splits = []
    for fold, (train_idx, val_idx) in enumerate(
        splitter.split(np.zeros(len(y)), y, groups=person_ids), start=1
    ):
        # Hard checks against person leakage.
        train_people = set(person_ids[train_idx])
        val_people = set(person_ids[val_idx])
        assert train_people.isdisjoint(val_people), f"Person leakage in repeat {repeat}, fold {fold}"
        assert set(record_ids[train_idx]).isdisjoint(set(record_ids[val_idx]))

        repeat_splits.append((train_idx, val_idx))
        split_rows.extend({
            "Repeat": repeat,
            "Fold": fold,
            "Row_Index": int(i),
            "Record_ID": int(record_ids[i]),
            "Person_ID": str(person_ids[i]),
            "AAMI_Label": str(y[i]),
            "Role": "validation"
        } for i in val_idx)

    split_indices[repeat] = repeat_splits

split_table = pd.DataFrame(split_rows)
split_table.to_csv("repeated_patient_grouped_splits.csv", index=False)
print("Saved repeated_patient_grouped_splits.csv")
print("Split table shape:", split_table.shape)
display(split_table.groupby(["Repeat", "Fold"]).agg(
    Validation_Rows=("Row_Index", "size"),
    Validation_People=("Person_ID", "nunique"),
    Validation_Records=("Record_ID", "nunique")
).reset_index())


Saved repeated_patient_grouped_splits.csv
Split table shape: (525235, 7)


,Repeat,Fold,Validation_Rows,Validation_People,Validation_Records
0,1,1,20414,9,9
1,1,2,21851,8,8
2,1,3,19942,9,9
3,1,4,23497,10,11
4,1,5,19343,9,9
5,2,1,19997,9,9
6,2,2,18214,8,8
7,2,3,20893,9,9
8,2,4,22866,10,10
9,2,5,23077,9,10


In [11]:
# 9. One evaluation function for all runs
# Saves one row per evaluated beat, one row per class per fold, fold-level metrics,
# and confusion matrices. Probability columns always follow CLASS_ORDER.

def make_rf(n_estimators, seed=SEED):
    return RandomForestClassifier(
        n_estimators=n_estimators,
        min_samples_split=2,
        min_samples_leaf=1,
        max_features="log2",
        max_depth=None,
        random_state=seed,
        n_jobs=-1,
        class_weight=None
    )

def aligned_probabilities(model, X_val):
    raw_prob = model.predict_proba(X_val)
    out = np.zeros((len(X_val), len(CLASS_ORDER)), dtype=float)
    model_classes = list(model.classes_)
    for j, cls in enumerate(CLASS_ORDER):
        if cls in model_classes:
            out[:, j] = raw_prob[:, model_classes.index(cls)]
    return out

def evaluate_pipeline(run_name, feature_kind, use_anova=False, use_ros=False, n_estimators=200):
    prediction_rows = []
    per_class_rows = []
    confusion_rows = []
    fold_metric_rows = []

    for repeat, repeat_splits in split_indices.items():
        for fold, (train_idx, val_idx) in enumerate(repeat_splits, start=1):
            y_train, y_val = y[train_idx], y[val_idx]

            if feature_kind == "raw":
                X_train = aami_df.iloc[train_idx][mlii_cols + lead2_cols].to_numpy(dtype=np.float32)
                X_val = aami_df.iloc[val_idx][mlii_cols + lead2_cols].to_numpy(dtype=np.float32)
            elif feature_kind == "derivative":
                X_train = DERIVATIVE_FEATURES.iloc[train_idx].to_numpy(dtype=np.float32)
                X_val = DERIVATIVE_FEATURES.iloc[val_idx].to_numpy(dtype=np.float32)
            else:
                raise ValueError("feature_kind must be 'raw' or 'derivative'")

            # Fit ANOVA feature selection only on this fold's training data.
            selected_indices = None
            if use_anova:
                selector = SelectKBest(score_func=f_classif, k=8)
                X_train = selector.fit_transform(X_train, y_train)
                X_val = selector.transform(X_val)
                selected_indices = selector.get_support(indices=True).tolist()

            # Oversample only the training partition.
            if use_ros:
                try:
                    from imblearn.over_sampling import RandomOverSampler
                except ImportError as e:
                    raise ImportError("Install imbalanced-learn in the Kaggle environment to run P3.") from e
                ros = RandomOverSampler(random_state=SEED)
                X_fit, y_fit = ros.fit_resample(X_train, y_train)
            else:
                X_fit, y_fit = X_train, y_train

            model = make_rf(n_estimators=n_estimators, seed=SEED)
            model.fit(X_fit, y_fit)
            y_pred = model.predict(X_val)
            probs = aligned_probabilities(model, X_val)

            acc = accuracy_score(y_val, y_pred)
            bal_acc = balanced_accuracy_score(y_val, y_pred)
            macro_f1 = f1_score(y_val, y_pred, labels=CLASS_ORDER, average="macro", zero_division=0)
            fold_metric_rows.append({
                "Run": run_name, "Repeat": repeat, "Fold": fold,
                "Accuracy": acc, "Balanced_Accuracy": bal_acc, "Macro_F1": macro_f1,
                "Train_Rows_Before_Balancing": len(train_idx),
                "Train_Rows_After_Balancing": len(y_fit),
                "Validation_Rows": len(val_idx),
                "Selected_Feature_Indices": json.dumps(selected_indices) if selected_indices is not None else None
            })

            precision, recall, f1s, supports = precision_recall_fscore_support(
                y_val, y_pred, labels=CLASS_ORDER, zero_division=0
            )
            for i, cls in enumerate(CLASS_ORDER):
                per_class_rows.append({
                    "Run": run_name, "Repeat": repeat, "Fold": fold, "Class": cls,
                    "Precision": precision[i], "Recall": recall[i], "F1": f1s[i], "Support": int(supports[i])
                })

            cm = confusion_matrix(y_val, y_pred, labels=CLASS_ORDER)
            for i, true_cls in enumerate(CLASS_ORDER):
                for j, pred_cls in enumerate(CLASS_ORDER):
                    confusion_rows.append({
                        "Run": run_name, "Repeat": repeat, "Fold": fold,
                        "True_Class": true_cls, "Predicted_Class": pred_cls,
                        "Count": int(cm[i, j])
                    })

            for k, row_idx in enumerate(val_idx):
                item = {
                    "Run": run_name, "Repeat": repeat, "Fold": fold,
                    "Row_Index": int(row_idx),
                    "Record_ID": int(record_ids[row_idx]),
                    "Person_ID": str(person_ids[row_idx]),
                    "True_Label": str(y_val[k]),
                    "Predicted_Label": str(y_pred[k])
                }
                for j, cls in enumerate(CLASS_ORDER):
                    item[f"Prob_{cls}"] = float(probs[k, j])
                prediction_rows.append(item)

    predictions = pd.DataFrame(prediction_rows)
    per_class = pd.DataFrame(per_class_rows)
    confusion = pd.DataFrame(confusion_rows)
    fold_metrics = pd.DataFrame(fold_metric_rows)

    # Mean across all 25 repeat/fold evaluations; this is not a pooled OOF metric.
    summary = fold_metrics.groupby("Run")[["Accuracy", "Balanced_Accuracy", "Macro_F1"]].agg(["mean", "std"]).reset_index()
    return {
        "predictions": predictions,
        "per_class": per_class,
        "confusion": confusion,
        "fold_metrics": fold_metrics,
        "summary": summary
    }


10. Run P1–P3
- P1: 402 raw signal values, no selection, no balancing, Random Forest with 100 trees and seed 42.
- P2: 16 derivative features, ANOVA top 8 fitted within each training fold, no balancing, fixed Random Forest with 200 trees.
- P3: same as P2, but random oversampling is applied only to each training fold.
These runs use the exact same saved repeated patient-grouped splits. They may take time because the dataset is large and this evaluates 25 folds per pipeline.

In [12]:
# Run P1, P2, P3. Set RUN_EXPERIMENTS=False if you only want to inspect setup first.
RUN_EXPERIMENTS = True

results = {}
if RUN_EXPERIMENTS:
    results["P1"] = evaluate_pipeline(
        run_name="P1_raw_RF100", feature_kind="raw",
        use_anova=False, use_ros=False, n_estimators=100
    )
    results["P2"] = evaluate_pipeline(
        run_name="P2_derivative_ANOVA8_RF200", feature_kind="derivative",
        use_anova=True, use_ros=False, n_estimators=200
    )
    results["P3"] = evaluate_pipeline(
        run_name="P3_derivative_ANOVA8_ROS_RF200", feature_kind="derivative",
        use_anova=True, use_ros=True, n_estimators=200
    )

    all_fold_metrics = pd.concat([results[k]["fold_metrics"] for k in results], ignore_index=True)
    all_per_class = pd.concat([results[k]["per_class"] for k in results], ignore_index=True)
    all_confusion = pd.concat([results[k]["confusion"] for k in results], ignore_index=True)
    all_predictions = pd.concat([results[k]["predictions"] for k in results], ignore_index=True)

    all_fold_metrics.to_csv("phase1_fold_metrics.csv", index=False)
    all_per_class.to_csv("phase1_per_class_metrics.csv", index=False)
    all_confusion.to_csv("phase1_confusion_matrices.csv", index=False)
    all_predictions.to_csv("phase1_predictions_and_probabilities.csv", index=False)

    summary_rows = []
    for run_name, run_result in results.items():
        fm = run_result["fold_metrics"]
        summary_rows.append({
            "Run": run_name,
            "Accuracy_Mean": fm["Accuracy"].mean(),
            "Accuracy_SD": fm["Accuracy"].std(),
            "Balanced_Accuracy_Mean": fm["Balanced_Accuracy"].mean(),
            "Balanced_Accuracy_SD": fm["Balanced_Accuracy"].std(),
            "Macro_F1_Mean": fm["Macro_F1"].mean(),
            "Macro_F1_SD": fm["Macro_F1"].std()
        })
    phase1_summary = pd.DataFrame(summary_rows)
    phase1_summary.to_csv("phase1_summary.csv", index=False)
    display(phase1_summary)

    print("Saved:")
    for filename in [
        "phase1_fold_metrics.csv",
        "phase1_per_class_metrics.csv",
        "phase1_confusion_matrices.csv",
        "phase1_predictions_and_probabilities.csv",
        "phase1_summary.csv"
    ]:
        print(" -", filename)
else:
    print("Experiments skipped. Set RUN_EXPERIMENTS=True to run P1-P3.")


/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/usr/local/lib/python3.13/dist-packages/sklearn/metrics

,Run,Accuracy_Mean,Accuracy_SD,Balanced_Accuracy_Mean,Balanced_Accuracy_SD,Macro_F1_Mean,Macro_F1_SD
0,P1,0.861260,0.054697,0.384649,0.059606,0.332898,0.073787
1,P2,0.878417,0.046153,0.343157,0.075387,0.324573,0.078111
2,P3,0.868370,0.052334,0.334084,0.073514,0.315314,0.076687


Saved:
 - phase1_fold_metrics.csv
 - phase1_per_class_metrics.csv
 - phase1_confusion_matrices.csv
 - phase1_predictions_and_probabilities.csv
 - phase1_summary.csv


In [13]:
# 11. Inspect class-wise metrics and confusion matrices before proceeding
if RUN_EXPERIMENTS:
    print("Mean per-class metrics across the 25 repeat/fold evaluations:")
    display(all_per_class.groupby(["Run", "Class"])[["Precision", "Recall", "F1", "Support"]].mean().reset_index())

    # Example: inspect all five confusion matrices for one run.
    RUN_TO_INSPECT = "P2_derivative_ANOVA8_RF200"
    display(
        all_confusion[all_confusion["Run"] == RUN_TO_INSPECT]
        .sort_values(["Repeat", "Fold", "True_Class", "Predicted_Class"])
        .head(25)
    )


Mean per-class metrics across the 25 repeat/fold evaluations:


,Run,Class,Precision,Recall,F1,Support
0,P1_raw_RF100,F,0.203055,0.027633,0.045314,160.4
1,P1_raw_RF100,N,0.933635,0.926184,0.928798,18068.2
2,P1_raw_RF100,Q,0.413557,0.193220,0.201294,778.8
3,P1_raw_RF100,S,0.155669,0.004700,0.008885,556.2
4,P1_raw_RF100,V,0.415568,0.687098,0.480199,1445.8
5,P2_derivative_ANOVA8_RF200,F,0.014864,0.013860,0.013840,160.4
6,P2_derivative_ANOVA8_RF200,N,0.907316,0.960935,0.932671,18068.2
7,P2_derivative_ANOVA8_RF200,Q,0.380530,0.172321,0.168710,778.8
8,P2_derivative_ANOVA8_RF200,S,0.024235,0.001159,0.001275,556.2
9,P2_derivative_ANOVA8_RF200,V,0.551502,0.500259,0.506369,1445.8


,Run,Repeat,Fold,True_Class,Predicted_Class,Count
643,P2_derivative_ANOVA8_RF200,1,1,F,F,0
640,P2_derivative_ANOVA8_RF200,1,1,F,N,341
644,P2_derivative_ANOVA8_RF200,1,1,F,Q,0
641,P2_derivative_ANOVA8_RF200,1,1,F,S,0
642,P2_derivative_ANOVA8_RF200,1,1,F,V,45
628,P2_derivative_ANOVA8_RF200,1,1,N,F,2
625,P2_derivative_ANOVA8_RF200,1,1,N,N,16402
629,P2_derivative_ANOVA8_RF200,1,1,N,Q,12
626,P2_derivative_ANOVA8_RF200,1,1,N,S,220
627,P2_derivative_ANOVA8_RF200,1,1,N,V,73


In [14]:
# Check which recordings are present in our dataset
record_ids = sorted(aami_df["Record_ID"].unique())

print("Number of recordings:", len(record_ids))
print("Record IDs:")
print(record_ids)

# Check the two recordings known to belong to the same person
print("\nRecords 201 and 202:")
print(aami_df[aami_df["Record_ID"].isin([201, 202])]
      .groupby("Record_ID")
      .size())

Number of recordings: 46
Record IDs:
[np.int64(100), np.int64(101), np.int64(103), np.int64(105), np.int64(106), np.int64(107), np.int64(108), np.int64(109), np.int64(111), np.int64(112), np.int64(113), np.int64(114), np.int64(115), np.int64(116), np.int64(117), np.int64(118), np.int64(119), np.int64(121), np.int64(122), np.int64(123), np.int64(124), np.int64(200), np.int64(201), np.int64(202), np.int64(203), np.int64(205), np.int64(207), np.int64(208), np.int64(209), np.int64(210), np.int64(212), np.int64(213), np.int64(214), np.int64(215), np.int64(217), np.int64(219), np.int64(220), np.int64(221), np.int64(222), np.int64(223), np.int64(228), np.int64(230), np.int64(231), np.int64(232), np.int64(233), np.int64(234)]

Records 201 and 202:
Record_ID
201    1963
202    2136
dtype: int64


In [15]:
# Check whether the dataset contains a patient/subject ID column
print(aami_df.columns.tolist())

['Record_ID', 'MLII_0', 'MLII_1', 'MLII_2', 'MLII_3', 'MLII_4', 'MLII_5', 'MLII_6', 'MLII_7', 'MLII_8', 'MLII_9', 'MLII_10', 'MLII_11', 'MLII_12', 'MLII_13', 'MLII_14', 'MLII_15', 'MLII_16', 'MLII_17', 'MLII_18', 'MLII_19', 'MLII_20', 'MLII_21', 'MLII_22', 'MLII_23', 'MLII_24', 'MLII_25', 'MLII_26', 'MLII_27', 'MLII_28', 'MLII_29', 'MLII_30', 'MLII_31', 'MLII_32', 'MLII_33', 'MLII_34', 'MLII_35', 'MLII_36', 'MLII_37', 'MLII_38', 'MLII_39', 'MLII_40', 'MLII_41', 'MLII_42', 'MLII_43', 'MLII_44', 'MLII_45', 'MLII_46', 'MLII_47', 'MLII_48', 'MLII_49', 'MLII_50', 'MLII_51', 'MLII_52', 'MLII_53', 'MLII_54', 'MLII_55', 'MLII_56', 'MLII_57', 'MLII_58', 'MLII_59', 'MLII_60', 'MLII_61', 'MLII_62', 'MLII_63', 'MLII_64', 'MLII_65', 'MLII_66', 'MLII_67', 'MLII_68', 'MLII_69', 'MLII_70', 'MLII_71', 'MLII_72', 'MLII_73', 'MLII_74', 'MLII_75', 'MLII_76', 'MLII_77', 'MLII_78', 'MLII_79', 'MLII_80', 'MLII_81', 'MLII_82', 'MLII_83', 'MLII_84', 'MLII_85', 'MLII_86', 'MLII_87', 'MLII_88', 'MLII_89', 'MLII_

In [16]:
# Check how Person_ID was assigned to each recording
mapping = (
    aami_df[["Record_ID", "Person_ID"]]
    .drop_duplicates()
    .sort_values("Record_ID")
)

print("Unique people:", mapping["Person_ID"].nunique())
print("\nRecord-to-person mapping:")
print(mapping.to_string(index=False))

print("\nRecords 201 and 202:")
print(mapping[mapping["Record_ID"].isin([201, 202])])

Unique people: 45

Record-to-person mapping:
 Record_ID      Person_ID
       100            100
       101            101
       103            103
       105            105
       106            106
       107            107
       108            108
       109            109
       111            111
       112            112
       113            113
       114            114
       115            115
       116            116
       117            117
       118            118
       119            119
       121            121
       122            122
       123            123
       124            124
       200            200
       201 person_201_202
       202 person_201_202
       203            203
       205            205
       207            207
       208            208
       209            209
       210            210
       212            212
       213            213
       214            214
       215            215
       217            217
       219         

In [17]:
# Check the saved validation splits
for repeat, folds in split_indices.items():
    for fold, (train_idx, val_idx) in enumerate(folds, start=1):
        val_records = set(aami_df.iloc[val_idx]["Record_ID"])
        if 201 in val_records or 202 in val_records:
            print(
                f"Repeat {repeat}, Fold {fold}: "
                f"201 in validation = {201 in val_records}, "
                f"202 in validation = {202 in val_records}"
            )

Repeat 1, Fold 4: 201 in validation = True, 202 in validation = True
Repeat 2, Fold 5: 201 in validation = True, 202 in validation = True
Repeat 3, Fold 4: 201 in validation = True, 202 in validation = True
Repeat 4, Fold 4: 201 in validation = True, 202 in validation = True
Repeat 5, Fold 4: 201 in validation = True, 202 in validation = True


In [18]:
# Count each AAMI class in the four recordings
records_to_check = [208, 213, 232, 209]

counts = pd.crosstab(
    aami_df.loc[aami_df["Record_ID"].isin(records_to_check), "Record_ID"],
    aami_df.loc[aami_df["Record_ID"].isin(records_to_check), "AAMI_Label"]
)

counts = counts.reindex(columns=["N", "S", "V", "F", "Q"], fill_value=0)

print(counts)

AAMI_Label     N     S    V    F  Q
Record_ID                          
208         1585     2  992  372  2
209         2621   383    1    0  0
213         2639    28  220  362  0
232          398  1382    0    0  0


In [19]:
# Compare actual and predicted classes on difficult recordings
records_to_check = [208, 209, 213, 232]

for run_name in all_predictions["Run"].unique():
    print(f"\n===== {run_name} =====")

    subset = all_predictions[
        all_predictions["Record_ID"].isin(records_to_check)
    ]

    for record_id in records_to_check:
        record_data = subset[subset["Record_ID"] == record_id]

        print(f"\nRecord {record_id}")
        print(pd.crosstab(
            record_data["True_Label"],
            record_data["Predicted_Label"],
            normalize="index"
        ).round(3))
        


===== P1_raw_RF100 =====

Record 208
Predicted_Label      N      Q      S      V
True_Label                                 
F                0.759  0.002  0.001  0.239
N                0.994  0.000  0.003  0.003
Q                0.933  0.000  0.000  0.067
S                1.000  0.000  0.000  0.000
V                0.206  0.000  0.004  0.790

Record 209
Predicted_Label    F      N      S    V
True_Label                             
N                0.0  1.000  0.000  0.0
S                0.0  0.996  0.004  0.0
V                0.0  0.000  0.000  1.0

Record 213
Predicted_Label      F      N      Q      V
True_Label                                 
F                0.014  0.857  0.006  0.123
N                0.000  0.915  0.011  0.074
S                0.000  0.738  0.150  0.112
V                0.017  0.619  0.000  0.363

Record 232
Predicted_Label      N      S      V
True_Label                          
N                0.975  0.001  0.024
S                0.949  0.001  0.050

=====

In [20]:
# Check whether the three pipelines really have different predictions

print("Prediction counts by pipeline:")
print(all_predictions.groupby("Run")["Predicted_Label"].value_counts())

print("\nNumber of prediction rows per pipeline:")
print(all_predictions.groupby("Run").size())

# Compare P1, P2, and P3 on the same evaluated beats
pred_wide = all_predictions.pivot(
    index=["Repeat", "Fold", "Row_Index"],
    columns="Run",
    values="Predicted_Label"
)

print("\nFraction of predictions that differ:")
for a, b in [
    ("P1_raw_RF100", "P2_derivative_ANOVA8_RF200"),
    ("P1_raw_RF100", "P3_derivative_ANOVA8_ROS_RF200"),
    ("P2_derivative_ANOVA8_RF200", "P3_derivative_ANOVA8_ROS_RF200")
]:
    print(f"{a} vs {b}: {(pred_wide[a] != pred_wide[b]).mean():.4%}")

Prediction counts by pipeline:
Run                             Predicted_Label
P1_raw_RF100                    N                  447910
                                V                   68456
                                Q                    8586
                                F                     151
                                S                     132
P2_derivative_ANOVA8_RF200      N                  477656
                                V                   34402
                                Q                   10102
                                S                    2590
                                F                     485
P3_derivative_ANOVA8_ROS_RF200  N                  473574
                                V                   37708
                                Q                    9828
                                S                    3449
                                F                     676
Name: count, dtype: int64

Number of prediction row

In [21]:
# Compare P1, P2 and P3 directly on the four recordings
records_to_check = [208, 209, 213, 232]

for record_id in records_to_check:
    print(f"\nRecord {record_id}")

    sub = all_predictions[
        all_predictions["Record_ID"] == record_id
    ]

    print(
        sub.groupby(["Run", "True_Label", "Predicted_Label"])
           .size()
           .unstack(fill_value=0)
           .to_string()
    )


Record 208
Predicted_Label                               N  Q   S     V
Run                            True_Label                   
P1_raw_RF100                   F           1355  4   0   501
                               N           7910  2   7     6
                               Q              8  0   0     2
                               S             10  0   0     0
                               V            269  4   0  4687
P2_derivative_ANOVA8_RF200     F           1471  2   3   384
                               N           7865  2  24    34
                               Q             10  0   0     0
                               S             10  0   0     0
                               V           1446  0  31  3483
P3_derivative_ANOVA8_ROS_RF200 F           1408  4   1   447
                               N           7849  3  44    29
                               Q             10  0   0     0
                               S             10  0   0     0
            

In [22]:
# Verify the person mapping for our 46 retained recordings
mapping = (
    aami_df[["Record_ID", "Person_ID"]]
    .drop_duplicates()
    .sort_values("Record_ID")
    .reset_index(drop=True)
)

assert len(mapping) == 46, "Expected 46 recordings"
assert mapping["Record_ID"].nunique() == 46
assert mapping["Person_ID"].nunique() == 45

# Confirm records 201 and 202 share one person ID
person_201 = mapping.loc[
    mapping["Record_ID"] == 201, "Person_ID"
].iloc[0]

person_202 = mapping.loc[
    mapping["Record_ID"] == 202, "Person_ID"
].iloc[0]

assert person_201 == person_202

mapping.to_csv("record_to_person_mapping.csv", index=False)

print("Verified: 46 recordings, 45 people")
print("Records 201 and 202 share the same Person_ID.")
print("Saved record_to_person_mapping.csv")

Verified: 46 recordings, 45 people
Records 201 and 202 share the same Person_ID.
Saved record_to_person_mapping.csv


In [23]:
# Verify that no person appears in both training and validation
for repeat, folds in split_indices.items():
    for fold, (train_idx, val_idx) in enumerate(folds, start=1):
        train_people = set(aami_df.iloc[train_idx]["Person_ID"])
        val_people = set(aami_df.iloc[val_idx]["Person_ID"])

        assert train_people.isdisjoint(val_people), (
            f"Leakage found in repeat {repeat}, fold {fold}"
        )

print("PASS: No person-level overlap in any of the saved folds.")

PASS: No person-level overlap in any of the saved folds.


In [24]:

# ============================================================
# CHECKPOINT SAVE — MIT-BIH Phase 1
# Saves important variables so we can resume later.
# ============================================================

import os
import pickle
from datetime import datetime

checkpoint = {
    "checkpoint_name": "MIT-BIH Phase 1",
    "created_at": datetime.now().isoformat(),
    "objects": {}
}

# Essential variables for continuing the project
essential_variables = [
    "aami_df",
    "split_indices",
    "mapping",
    "record_to_person_mapping",
    "phase1_summary",
    "phase1_fold_metrics",
    "phase1_per_class_metrics",
    "phase1_confusion_matrices",
    "phase1_predictions_and_probabilities",
    "all_predictions",
    "all_per_class",
    "all_confusion",
]

# Save essential variables that currently exist
for variable_name in essential_variables:
    if variable_name in globals():
        checkpoint["objects"][variable_name] = globals()[variable_name]

# Also save any other variables whose names start with phase1_
for variable_name, variable_value in list(globals().items()):
    if variable_name.startswith("phase1_"):
        try:
            pickle.dumps(variable_value)
            checkpoint["objects"][variable_name] = variable_value
        except Exception:
            print(f"Skipped non-saveable variable: {variable_name}")

# Save the checkpoint file
checkpoint_path = "/kaggle/working/phase1_checkpoint.pkl"

with open(checkpoint_path, "wb") as file:
    pickle.dump(checkpoint, file, protocol=pickle.HIGHEST_PROTOCOL)

print("Checkpoint saved successfully!")
print("File:", checkpoint_path)
print("Saved variables:", list(checkpoint["objects"].keys()))
print(f"File size: {os.path.getsize(checkpoint_path) / (1024**2):.2f} MB")


Checkpoint saved successfully!
File: /kaggle/working/phase1_checkpoint.pkl
Saved variables: ['aami_df', 'split_indices', 'mapping', 'phase1_summary', 'all_predictions', 'all_per_class', 'all_confusion']
File size: 473.35 MB


In [25]:

import pickle
import os

checkpoint_path = "/kaggle/working/phase1_checkpoint.pkl"

with open(checkpoint_path, "rb") as f:
    saved_checkpoint = pickle.load(f)

saved = saved_checkpoint["objects"]

print("Checkpoint readable: YES")
print("Dataset shape:", saved["aami_df"].shape)
print("Number of CV repeats:", len(saved["split_indices"]))
print("Number of mapped records:", len(saved["mapping"]))
print("Saved variables:", list(saved.keys()))

# Verify that every repeat contains five folds
for repeat, folds in saved["split_indices"].items():
    assert len(folds) == 5, f"Repeat {repeat} does not have 5 folds."

print("PASS: All saved repeats contain five folds.")
print(f"Checkpoint size: {os.path.getsize(checkpoint_path) / (1024**2):.2f} MB")


Checkpoint readable: YES
Dataset shape: (105047, 406)
Number of CV repeats: 5
Number of mapped records: 46
Saved variables: ['aami_df', 'split_indices', 'mapping', 'phase1_summary', 'all_predictions', 'all_per_class', 'all_confusion']
PASS: All saved repeats contain five folds.
Checkpoint size: 473.35 MB


In [26]:

# Verify patient-level separation in every saved fold

saved_df = saved["aami_df"]
saved_splits = saved["split_indices"]

for repeat, folds in saved_splits.items():
    for fold_number, (train_idx, val_idx) in enumerate(folds, start=1):
        train_people = set(saved_df.iloc[train_idx]["Person_ID"])
        val_people = set(saved_df.iloc[val_idx]["Person_ID"])

        assert train_people.isdisjoint(val_people), (
            f"Patient leakage detected in repeat {repeat}, fold {fold_number}"
        )

print("PASS: No patient-level overlap in all saved folds.")


PASS: No patient-level overlap in all saved folds.


In [27]:

# Check which Phase 2 inputs are currently available

candidate_names = [
    "aami_df",
    "split_indices",
    "mapping",
    "DERIVATIVE_FEATURES",
    "X",
    "y",
    "record_ids",
    "person_ids",
]

for name in candidate_names:
    if name in globals():
        obj = globals()[name]
        print(f"{name}: available — {type(obj).__name__}")
    else:
        print(f"{name}: not currently available")


aami_df: available — DataFrame
split_indices: available — dict
mapping: available — DataFrame
DERIVATIVE_FEATURES: available — DataFrame
X: not currently available
y: available — ndarray
record_ids: available — list
person_ids: available — ndarray


In [28]:

print("DERIVATIVE_FEATURES shape:", DERIVATIVE_FEATURES.shape)
print("\nFirst 20 feature columns:")
print(DERIVATIVE_FEATURES.columns[:20].tolist())

print("\naami_df shape:", aami_df.shape)
print("\nLabels available:")
print(aami_df["AAMI_Label"].value_counts().sort_index())

print("\nIndex alignment check:")
print("Same number of rows:", len(DERIVATIVE_FEATURES) == len(aami_df))
print("Same index:", DERIVATIVE_FEATURES.index.equals(aami_df.index))


DERIVATIVE_FEATURES shape: (105047, 16)

First 20 feature columns:
['MLII_diff_mean', 'MLII_diff_std', 'MLII_diff_max', 'MLII_diff_min', 'MLII_diff_abs_mean', 'MLII_diff_energy', 'MLII_second_diff_std', 'MLII_second_diff_abs_mean', 'Lead2_diff_mean', 'Lead2_diff_std', 'Lead2_diff_max', 'Lead2_diff_min', 'Lead2_diff_abs_mean', 'Lead2_diff_energy', 'Lead2_second_diff_std', 'Lead2_second_diff_abs_mean']

aami_df shape: (105047, 406)

Labels available:
AAMI_Label
F      802
N    90341
Q     3894
S     2781
V     7229
Name: count, dtype: int64

Index alignment check:
Same number of rows: True
Same index: True


In [29]:

print("Columns that may contain beat timing information:")

timing_candidates = [
    col for col in aami_df.columns
    if any(term in col.lower() for term in [
        "time", "timestamp", "sample", "beat_index",
        "annotation", "position", "r_peak"
    ])
]

print(timing_candidates)

print("\nFirst five rows of record and label information:")
print(
    aami_df[["Record_ID", "Label", "AAMI_Label"]].head()
)


Columns that may contain beat timing information:
[]

First five rows of record and label information:
   Record_ID Label AAMI_Label
0        100     N          N
1        100     N          N
2        100     N          N
3        100     N          N
4        100     N          N


In [31]:

import os

for root, dirs, files in os.walk("/kaggle/input"):
    matching_files = [
        f for f in files
        if f.lower().endswith((".atr", ".hea", ".dat"))
    ]

    if matching_files:
        print(f"\nFolder: {root}")
        print("Example files:", matching_files[:12])


In [32]:

import os

print("Files available in Kaggle input folders:\n")

for root, dirs, files in os.walk("/kaggle/input"):
    relative_root = os.path.relpath(root, "/kaggle/input")
    print(f"\nFolder: {relative_root}/")

    for filename in files[:15]:
        print("  ", filename)

    if len(files) > 15:
        print(f"   ... and {len(files) - 15} more files")


Files available in Kaggle input folders:


Folder: ./

Folder: datasets/

Folder: datasets/aakashshah3799/

Folder: datasets/aakashshah3799/mit-bih-own/
   MITBIH_MLII_DualLead_Dataset.csv
   mit_bih_stage7_final_checkpoint.pkl


In [33]:

import pandas as pd

csv_path = (
    "/kaggle/input/datasets/aakashshah3799/"
    "mit-bih-own/MITBIH_MLII_DualLead_Dataset.csv"
)

source_df = pd.read_csv(csv_path, nrows=5)

print("Original CSV columns:")
print(source_df.columns.tolist())

print("\nFirst five rows:")
display(source_df.head())

print("\nCurrent dataset row count:", len(aami_df))
print("Original CSV preview shape:", source_df.shape)


Original CSV columns:
['Record_ID', 'MLII_0', 'MLII_1', 'MLII_2', 'MLII_3', 'MLII_4', 'MLII_5', 'MLII_6', 'MLII_7', 'MLII_8', 'MLII_9', 'MLII_10', 'MLII_11', 'MLII_12', 'MLII_13', 'MLII_14', 'MLII_15', 'MLII_16', 'MLII_17', 'MLII_18', 'MLII_19', 'MLII_20', 'MLII_21', 'MLII_22', 'MLII_23', 'MLII_24', 'MLII_25', 'MLII_26', 'MLII_27', 'MLII_28', 'MLII_29', 'MLII_30', 'MLII_31', 'MLII_32', 'MLII_33', 'MLII_34', 'MLII_35', 'MLII_36', 'MLII_37', 'MLII_38', 'MLII_39', 'MLII_40', 'MLII_41', 'MLII_42', 'MLII_43', 'MLII_44', 'MLII_45', 'MLII_46', 'MLII_47', 'MLII_48', 'MLII_49', 'MLII_50', 'MLII_51', 'MLII_52', 'MLII_53', 'MLII_54', 'MLII_55', 'MLII_56', 'MLII_57', 'MLII_58', 'MLII_59', 'MLII_60', 'MLII_61', 'MLII_62', 'MLII_63', 'MLII_64', 'MLII_65', 'MLII_66', 'MLII_67', 'MLII_68', 'MLII_69', 'MLII_70', 'MLII_71', 'MLII_72', 'MLII_73', 'MLII_74', 'MLII_75', 'MLII_76', 'MLII_77', 'MLII_78', 'MLII_79', 'MLII_80', 'MLII_81', 'MLII_82', 'MLII_83', 'MLII_84', 'MLII_85', 'MLII_86', 'MLII_87', 'MLII_

,Record_ID,MLII_0,MLII_1,MLII_2,MLII_3,MLII_4,MLII_5,MLII_6,MLII_7,MLII_8,...,Lead2_192,Lead2_193,Lead2_194,Lead2_195,Lead2_196,Lead2_197,Lead2_198,Lead2_199,Lead2_200,Label
0,100,961,964,963,962,960,958,959,961,962,...,946,947,946,944,942,940,943,942,942,N
1,100,955,955,955,958,958,957,955,954,954,...,939,940,942,943,944,944,943,947,951,N
2,100,951,955,955,956,953,951,952,953,953,...,949,949,949,948,951,955,957,961,962,N
3,100,954,954,954,956,956,955,955,948,952,...,941,939,942,945,947,943,945,945,949,N
4,100,958,961,961,958,958,957,957,958,958,...,951,953,951,950,949,949,950,953,952,N



Current dataset row count: 105047
Original CSV preview shape: (5, 404)


In [34]:

# ============================================================
# PHASE 2 — F2: MLII-only derivative features
# ============================================================

# F1: Existing derivative features from both ECG leads
X_F1 = DERIVATIVE_FEATURES.copy()

# F2: Keep only features derived from the MLII lead
mlii_columns = [
    col for col in DERIVATIVE_FEATURES.columns
    if col.startswith("MLII_")
]

X_F2 = DERIVATIVE_FEATURES[mlii_columns].copy()

# Check alignment before any model training
assert X_F1.index.equals(aami_df.index)
assert X_F2.index.equals(aami_df.index)
assert len(X_F1) == len(X_F2) == len(aami_df)

print("F1 — Both leads:", X_F1.shape)
print("F2 — MLII only:", X_F2.shape)
print("\nMLII-only features:")
print(mlii_columns)

print("\nLabels:")
print(aami_df["AAMI_Label"].value_counts().sort_index())


F1 — Both leads: (105047, 16)
F2 — MLII only: (105047, 8)

MLII-only features:
['MLII_diff_mean', 'MLII_diff_std', 'MLII_diff_max', 'MLII_diff_min', 'MLII_diff_abs_mean', 'MLII_diff_energy', 'MLII_second_diff_std', 'MLII_second_diff_abs_mean']

Labels:
AAMI_Label
F      802
N    90341
Q     3894
S     2781
V     7229
Name: count, dtype: int64


In [35]:

# ============================================================
# PHASE 2 — Compare F1 (both leads) vs F2 (MLII only)
# Uses existing patient-grouped split_indices
# ============================================================

import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score
)

y_phase2 = aami_df["AAMI_Label"].to_numpy()

feature_sets = {
    "F1_both_leads": X_F1,
    "F2_MLII_only": X_F2,
}

phase2_fold_results = []
phase2_predictions = []

for feature_name, X_features in feature_sets.items():
    print(f"\nRunning {feature_name}...")

    X_array = X_features.to_numpy()

    for repeat, folds in split_indices.items():
        for fold_number, (train_idx, val_idx) in enumerate(folds, start=1):

            X_train = X_array[train_idx]
            X_val = X_array[val_idx]
            y_train = y_phase2[train_idx]
            y_val = y_phase2[val_idx]

            model = RandomForestClassifier(
                n_estimators=200,
                max_features="log2",
                min_samples_split=2,
                min_samples_leaf=1,
                random_state=42,
                n_jobs=-1
            )

            model.fit(X_train, y_train)
            y_pred = model.predict(X_val)

            phase2_fold_results.append({
                "Feature_Set": feature_name,
                "Repeat": repeat,
                "Fold": fold_number,
                "Accuracy": accuracy_score(y_val, y_pred),
                "Balanced_Accuracy": balanced_accuracy_score(y_val, y_pred),
                "Macro_F1": f1_score(
                    y_val, y_pred,
                    labels=["N", "S", "V", "F", "Q"],
                    average="macro",
                    zero_division=0
                )
            })

            # Save predictions with their original dataset row positions
            for i, prediction in zip(val_idx, y_pred):
                phase2_predictions.append({
                    "Feature_Set": feature_name,
                    "Repeat": repeat,
                    "Fold": fold_number,
                    "Row_Index": int(i),
                    "Record_ID": aami_df.iloc[i]["Record_ID"],
                    "Person_ID": aami_df.iloc[i]["Person_ID"],
                    "True_Label": y_phase2[i],
                    "Predicted_Label": prediction
                })

    print(f"Finished {feature_name}.")

# Store results as DataFrames
phase2_fold_metrics = pd.DataFrame(phase2_fold_results)
phase2_predictions_df = pd.DataFrame(phase2_predictions)

# Summarize the 25 fold scores for each feature set
phase2_summary = (
    phase2_fold_metrics
    .groupby("Feature_Set")
    .agg(
        Accuracy_Mean=("Accuracy", "mean"),
        Accuracy_SD=("Accuracy", "std"),
        Balanced_Accuracy_Mean=("Balanced_Accuracy", "mean"),
        Balanced_Accuracy_SD=("Balanced_Accuracy", "std"),
        Macro_F1_Mean=("Macro_F1", "mean"),
        Macro_F1_SD=("Macro_F1", "std")
    )
    .reset_index()
)

# Save results to Kaggle working storage
phase2_fold_metrics.to_csv(
    "/kaggle/working/phase2_fold_metrics.csv", index=False
)
phase2_predictions_df.to_csv(
    "/kaggle/working/phase2_predictions.csv", index=False
)
phase2_summary.to_csv(
    "/kaggle/working/phase2_summary.csv", index=False
)

print("\nPHASE 2 SUMMARY")
display(phase2_summary)

print("\nSaved files:")
print("/kaggle/working/phase2_fold_metrics.csv")
print("/kaggle/working/phase2_predictions.csv")
print("/kaggle/working/phase2_summary.csv")



Running F1_both_leads...


/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")


Finished F1_both_leads.

Running F2_MLII_only...


/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")


Finished F2_MLII_only.

PHASE 2 SUMMARY


,Feature_Set,Accuracy_Mean,Accuracy_SD,Balanced_Accuracy_Mean,Balanced_Accuracy_SD,Macro_F1_Mean,Macro_F1_SD
0,F1_both_leads,0.879840,0.045350,0.334017,0.062210,0.322155,0.074684
1,F2_MLII_only,0.875829,0.051709,0.368825,0.072969,0.342625,0.066422



Saved files:
/kaggle/working/phase2_fold_metrics.csv
/kaggle/working/phase2_predictions.csv
/kaggle/working/phase2_summary.csv


In [36]:

from sklearn.metrics import classification_report

class_names = ["N", "S", "V", "F", "Q"]

for feature_name in ["F1_both_leads", "F2_MLII_only"]:
    print(f"\n{'=' * 55}")
    print(feature_name)
    print("=" * 55)

    subset = phase2_predictions_df[
        phase2_predictions_df["Feature_Set"] == feature_name
    ]

    print(classification_report(
        subset["True_Label"],
        subset["Predicted_Label"],
        labels=class_names,
        target_names=class_names,
        zero_division=0,
        digits=4
    ))



F1_both_leads
              precision    recall  f1-score   support

           N     0.9066    0.9657    0.9353    451705
           S     0.0109    0.0013    0.0023     13905
           V     0.5437    0.5273    0.5354     36145
           F     0.0178    0.0010    0.0019      4010
           Q     0.9863    0.3624    0.5300     19470

    accuracy                         0.8803    525235
   macro avg     0.4931    0.3716    0.4010    525235
weighted avg     0.8541    0.8803    0.8609    525235


F2_MLII_only
              precision    recall  f1-score   support

           N     0.9185    0.9466    0.9324    451705
           S     0.0145    0.0039    0.0061     13905
           V     0.5343    0.6638    0.5920     36145
           F     0.0032    0.0010    0.0015      4010
           Q     0.8207    0.4128    0.5493     19470

    accuracy                         0.8752    525235
   macro avg     0.4582    0.4056    0.4163    525235
weighted avg     0.8575    0.8752    0.8631    5

In [38]:

import os
import pickle
from datetime import datetime

phase2_checkpoint = {
    "checkpoint_name": "MIT-BIH Phase 2",
    "created_at": datetime.now().isoformat(),
    "objects": {}
}

phase2_variables = [
    "aami_df",
    "split_indices",
    "mapping",
    "DERIVATIVE_FEATURES",
    "X_F1",
    "X_F2",
    "y_phase2",
    "phase2_fold_metrics",
    "phase2_predictions_df",
    "phase2_summary",
]

for name in phase2_variables:
    if name in globals():
        phase2_checkpoint["objects"][name] = globals()[name]

checkpoint_path = "/kaggle/working/phase2_checkpoint.pkl"

with open(checkpoint_path, "wb") as f:
    pickle.dump(
        phase2_checkpoint,
        f,
        protocol=pickle.HIGHEST_PROTOCOL
    )

print("Phase 2 checkpoint saved.")
print("Path:", checkpoint_path)
print(
    "Size (MB):",
    round(os.path.getsize(checkpoint_path) / (1024**2), 2)
)
print("Saved variables:", list(phase2_checkpoint["objects"].keys()))


Phase 2 checkpoint saved.
Path: /kaggle/working/phase2_checkpoint.pkl
Size (MB): 422.38
Saved variables: ['aami_df', 'split_indices', 'mapping', 'DERIVATIVE_FEATURES', 'X_F1', 'X_F2', 'y_phase2', 'phase2_fold_metrics', 'phase2_predictions_df', 'phase2_summary']


In [39]:

mlii_sample_cols = [
    f"MLII_{i}" for i in range(201)
]

print("MLII data types:")
print(aami_df[mlii_sample_cols].dtypes.value_counts())

print("\nMissing MLII values:", 
      int(aami_df[mlii_sample_cols].isna().sum().sum()))

print("\nExample waveform values:")
print(aami_df[mlii_sample_cols].iloc[0, :10].tolist())

print("\nMLII amplitude summary:")
print(aami_df[mlii_sample_cols].to_numpy().min(),
      aami_df[mlii_sample_cols].to_numpy().max())


MLII data types:
int64    201
Name: count, dtype: int64

Missing MLII values: 0

Example waveform values:
[961, 964, 963, 962, 960, 958, 959, 961, 962, 963]

MLII amplitude summary:
0 2047


In [40]:

import numpy as np
import pandas as pd

# Extract the 201 MLII samples per beat
mlii_cols = [f"MLII_{i}" for i in range(201)]
waveforms = aami_df[mlii_cols].to_numpy(dtype=np.float32)

# Normalize each beat independently:
# subtract its mean and divide by its standard deviation.
beat_means = waveforms.mean(axis=1, keepdims=True)
beat_stds = waveforms.std(axis=1, keepdims=True)

# Avoid division by zero for a flat waveform
beat_stds[beat_stds == 0] = 1.0

normalized_waveforms = (waveforms - beat_means) / beat_stds

# Calculate derivative-based features from normalized waveforms
first_diff = np.diff(normalized_waveforms, axis=1)
second_diff = np.diff(normalized_waveforms, n=2, axis=1)

X_F3 = pd.DataFrame({
    "MLII_diff_mean": first_diff.mean(axis=1),
    "MLII_diff_std": first_diff.std(axis=1),
    "MLII_diff_max": first_diff.max(axis=1),
    "MLII_diff_min": first_diff.min(axis=1),
    "MLII_diff_abs_mean": np.abs(first_diff).mean(axis=1),
    "MLII_diff_energy": np.square(first_diff).mean(axis=1),
    "MLII_second_diff_std": second_diff.std(axis=1),
    "MLII_second_diff_abs_mean": np.abs(second_diff).mean(axis=1),
}, index=aami_df.index)

# Alignment checks
assert X_F3.shape == (len(aami_df), 8)
assert X_F3.index.equals(aami_df.index)
assert np.isfinite(X_F3.to_numpy()).all()

print("F2 — Existing MLII derivatives:", X_F2.shape)
print("F3 — Normalized MLII derivatives:", X_F3.shape)
print("\nF3 feature preview:")
display(X_F3.head())
print("PASS: Feature rows are aligned and contain finite values.")


F2 — Existing MLII derivatives: (105047, 8)
F3 — Normalized MLII derivatives: (105047, 8)

F3 feature preview:


,MLII_diff_mean,MLII_diff_std,MLII_diff_max,MLII_diff_min,MLII_diff_abs_mean,MLII_diff_energy,MLII_second_diff_std,MLII_second_diff_abs_mean
0,-3.033295e-03,0.250179,1.145912,-1.460476,0.104143,0.062599,0.118295,0.074972
1,-1.957011e-03,0.320193,1.418833,-2.299488,0.115708,0.102528,0.177626,0.091458
2,-9.685754e-10,0.317766,1.446837,-2.118582,0.120914,0.100976,0.172947,0.091791
3,-2.598191e-03,0.322034,1.586264,-2.133252,0.124030,0.103712,0.180324,0.100739
4,-2.705870e-03,0.309085,1.443130,-2.010075,0.118156,0.095541,0.163158,0.091037


PASS: Feature rows are aligned and contain finite values.


In [41]:

# ============================================================
# PHASE 2 — F2 vs F3: effect of per-beat normalization
# ============================================================

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score
)
import pandas as pd
import numpy as np

y_phase2 = aami_df["AAMI_Label"].to_numpy()

feature_sets_norm = {
    "F2_MLII_original": X_F2,
    "F3_MLII_normalized": X_F3,
}

norm_fold_results = []
norm_predictions = []

for feature_name, X_features in feature_sets_norm.items():
    print(f"\nRunning {feature_name}...")

    X_array = X_features.to_numpy()

    for repeat, folds in split_indices.items():
        for fold_number, (train_idx, val_idx) in enumerate(folds, start=1):
            model = RandomForestClassifier(
                n_estimators=200,
                max_features="log2",
                min_samples_split=2,
                min_samples_leaf=1,
                random_state=42,
                n_jobs=-1
            )

            model.fit(X_array[train_idx], y_phase2[train_idx])
            y_pred = model.predict(X_array[val_idx])
            y_true = y_phase2[val_idx]

            norm_fold_results.append({
                "Feature_Set": feature_name,
                "Repeat": repeat,
                "Fold": fold_number,
                "Accuracy": accuracy_score(y_true, y_pred),
                "Balanced_Accuracy": balanced_accuracy_score(y_true, y_pred),
                "Macro_F1": f1_score(
                    y_true, y_pred,
                    labels=["N", "S", "V", "F", "Q"],
                    average="macro",
                    zero_division=0
                )
            })

            for i, pred in zip(val_idx, y_pred):
                norm_predictions.append({
                    "Feature_Set": feature_name,
                    "Repeat": repeat,
                    "Fold": fold_number,
                    "Row_Index": int(i),
                    "True_Label": y_phase2[i],
                    "Predicted_Label": pred
                })

    print(f"Finished {feature_name}.")

norm_fold_metrics = pd.DataFrame(norm_fold_results)
norm_predictions_df = pd.DataFrame(norm_predictions)

norm_summary = (
    norm_fold_metrics.groupby("Feature_Set")
    .agg(
        Accuracy_Mean=("Accuracy", "mean"),
        Accuracy_SD=("Accuracy", "std"),
        Balanced_Accuracy_Mean=("Balanced_Accuracy", "mean"),
        Balanced_Accuracy_SD=("Balanced_Accuracy", "std"),
        Macro_F1_Mean=("Macro_F1", "mean"),
        Macro_F1_SD=("Macro_F1", "std")
    )
    .reset_index()
)

norm_fold_metrics.to_csv(
    "/kaggle/working/phase2_normalization_fold_metrics.csv",
    index=False
)
norm_predictions_df.to_csv(
    "/kaggle/working/phase2_normalization_predictions.csv",
    index=False
)
norm_summary.to_csv(
    "/kaggle/working/phase2_normalization_summary.csv",
    index=False
)

print("\nNORMALIZATION COMPARISON")
display(norm_summary)
print("\nSaved fold metrics, predictions, and summary.")



Running F2_MLII_original...


/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")


Finished F2_MLII_original.

Running F3_MLII_normalized...


/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")


Finished F3_MLII_normalized.

NORMALIZATION COMPARISON


,Feature_Set,Accuracy_Mean,Accuracy_SD,Balanced_Accuracy_Mean,Balanced_Accuracy_SD,Macro_F1_Mean,Macro_F1_SD
0,F2_MLII_original,0.875829,0.051709,0.368825,0.072969,0.342625,0.066422
1,F3_MLII_normalized,0.900351,0.045870,0.432132,0.088040,0.395877,0.094875



Saved fold metrics, predictions, and summary.


In [42]:

import pickle
import os
from datetime import datetime

phase2_checkpoint_v2 = {
    "checkpoint_name": "MIT-BIH Phase 2 - Normalization Results",
    "created_at": datetime.now().isoformat(),
    "objects": {}
}

variables_to_save = [
    "aami_df",
    "split_indices",
    "mapping",
    "DERIVATIVE_FEATURES",
    "X_F1",
    "X_F2",
    "X_F3",
    "y_phase2",
    "phase2_fold_metrics",
    "phase2_predictions_df",
    "phase2_summary",
    "norm_fold_metrics",
    "norm_predictions_df",
    "norm_summary",
]

for name in variables_to_save:
    if name in globals():
        phase2_checkpoint_v2["objects"][name] = globals()[name]

checkpoint_path = "/kaggle/working/phase2_checkpoint_v2.pkl"

with open(checkpoint_path, "wb") as f:
    pickle.dump(
        phase2_checkpoint_v2,
        f,
        protocol=pickle.HIGHEST_PROTOCOL
    )

print("Checkpoint saved:", checkpoint_path)
print("Size (MB):", round(os.path.getsize(checkpoint_path) / (1024**2), 2))
print("Saved variables:", list(phase2_checkpoint_v2["objects"].keys()))


Checkpoint saved: /kaggle/working/phase2_checkpoint_v2.pkl
Size (MB): 458.65
Saved variables: ['aami_df', 'split_indices', 'mapping', 'DERIVATIVE_FEATURES', 'X_F1', 'X_F2', 'X_F3', 'y_phase2', 'phase2_fold_metrics', 'phase2_predictions_df', 'phase2_summary', 'norm_fold_metrics', 'norm_predictions_df', 'norm_summary']


In [43]:

from sklearn.metrics import classification_report
import pandas as pd

class_names = ["N", "S", "V", "F", "Q"]

for feature_name in ["F2_MLII_original", "F3_MLII_normalized"]:
    print(f"\n{'=' * 60}")
    print(feature_name)
    print("=" * 60)

    subset = norm_predictions_df[
        norm_predictions_df["Feature_Set"] == feature_name
    ]

    print(classification_report(
        subset["True_Label"],
        subset["Predicted_Label"],
        labels=class_names,
        target_names=class_names,
        zero_division=0,
        digits=4
    ))



F2_MLII_original
              precision    recall  f1-score   support

           N     0.9185    0.9466    0.9324    451705
           S     0.0145    0.0039    0.0061     13905
           V     0.5343    0.6638    0.5920     36145
           F     0.0032    0.0010    0.0015      4010
           Q     0.8207    0.4128    0.5493     19470

    accuracy                         0.8752    525235
   macro avg     0.4582    0.4056    0.4163    525235
weighted avg     0.8575    0.8752    0.8631    525235


F3_MLII_normalized
              precision    recall  f1-score   support

           N     0.9438    0.9445    0.9441    451705
           S     0.0093    0.0014    0.0025     13905
           V     0.5807    0.8087    0.6760     36145
           F     0.0024    0.0022    0.0023      4010
           Q     0.9743    0.8477    0.9066     19470

    accuracy                         0.8994    525235
   macro avg     0.5021    0.5209    0.5063    525235
weighted avg     0.8880    0.8994    0.

In [44]:

import pandas as pd
from sklearn.metrics import confusion_matrix

class_names = ["N", "S", "V", "F", "Q"]

subset = norm_predictions_df[
    norm_predictions_df["Feature_Set"] == "F3_MLII_normalized"
]

cm = confusion_matrix(
    subset["True_Label"],
    subset["Predicted_Label"],
    labels=class_names
)

cm_df = pd.DataFrame(
    cm,
    index=[f"Actual_{c}" for c in class_names],
    columns=[f"Predicted_{c}" for c in class_names]
)

print("Pooled confusion matrix for F3:")
display(cm_df)

# Row-normalized matrix: percentages within each true class
row_totals = cm.sum(axis=1, keepdims=True)
cm_percent = 100 * cm / row_totals.clip(min=1)

cm_percent_df = pd.DataFrame(
    cm_percent.round(2),
    index=[f"Actual_{c}" for c in class_names],
    columns=[f"Predicted_{c}" for c in class_names]
)

print("\nRow-normalized confusion matrix (%):")
display(cm_percent_df)


Pooled confusion matrix for F3:


,Predicted_N,Predicted_S,Predicted_V,Predicted_F,Predicted_Q
Actual_N,426620,1954,19226,3490,415
Actual_S,13317,20,476,92,0
Actual_V,6523,163,29229,212,18
Actual_F,2678,5,1316,9,2
Actual_Q,2870,7,89,0,16504



Row-normalized confusion matrix (%):


,Predicted_N,Predicted_S,Predicted_V,Predicted_F,Predicted_Q
Actual_N,94.45,0.43,4.26,0.77,0.09
Actual_S,95.77,0.14,3.42,0.66,0.00
Actual_V,18.05,0.45,80.87,0.59,0.05
Actual_F,66.78,0.12,32.82,0.22,0.05
Actual_Q,14.74,0.04,0.46,0.00,84.77


In [45]:

import pandas as pd
import numpy as np

# Use saved F3 predictions; no model retraining required
f3 = norm_predictions_df[
    norm_predictions_df["Feature_Set"] == "F3_MLII_normalized"
].copy()

# Recover record IDs using the original dataset row indices
f3["Record_ID"] = (
    aami_df["Record_ID"].to_numpy()[f3["Row_Index"].to_numpy()]
)

# Keep only actual S and F beats
rare = f3[f3["True_Label"].isin(["S", "F"])].copy()

record_diagnostics = (
    rare.groupby(["Record_ID", "True_Label"])
    .agg(
        Actual_Beats=("True_Label", "size"),
        Correct_Predictions=(
            "Predicted_Label",
            lambda s: (s == rare.loc[s.index, "True_Label"]).sum()
        )
    )
    .reset_index()
)

record_diagnostics["Recall"] = (
    record_diagnostics["Correct_Predictions"]
    / record_diagnostics["Actual_Beats"]
)

print("S and F performance by recording:")
display(
    record_diagnostics.sort_values(
        ["True_Label", "Recall", "Actual_Beats"],
        ascending=[True, True, False]
    )
)

record_diagnostics.to_csv(
    "/kaggle/working/phase2_F3_rare_class_record_diagnostics.csv",
    index=False
)

print("\nSaved: phase2_F3_rare_class_record_diagnostics.csv")


S and F performance by recording:


,Record_ID,True_Label,Actual_Beats,Correct_Predictions,Recall
27,208,F,1860,0,0.000000
32,213,F,1810,0,0.000000
46,233,F,55,0,0.000000
30,210,F,50,0,0.000000
14,124,F,25,0,0.000000
8,114,F,20,0,0.000000
3,108,F,10,0,0.000000
5,109,F,10,0,0.000000
16,200,F,10,0,0.000000
18,201,F,10,0,0.000000



Saved: phase2_F3_rare_class_record_diagnostics.csv


In [46]:

# ============================================================
# PHASE 2 — Test random oversampling on normalized MLII features
# Oversampling is applied to training data only.
# ============================================================

import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score
)
from imblearn.over_sampling import RandomOverSampler

X_array = X_F3.to_numpy()
y_array = aami_df["AAMI_Label"].to_numpy()
class_names = ["N", "S", "V", "F", "Q"]

balance_results = []
balance_predictions = []

for setting in ["F3_no_oversampling", "F3_random_oversampling"]:
    print(f"\nRunning {setting}...")

    for repeat, folds in split_indices.items():
        for fold_number, (train_idx, val_idx) in enumerate(folds, start=1):

            X_train = X_array[train_idx]
            y_train = y_array[train_idx]
            X_val = X_array[val_idx]
            y_val = y_array[val_idx]

            if setting == "F3_random_oversampling":
                sampler = RandomOverSampler(random_state=42)
                X_train, y_train = sampler.fit_resample(X_train, y_train)

            model = RandomForestClassifier(
                n_estimators=200,
                max_features="log2",
                min_samples_split=2,
                min_samples_leaf=1,
                random_state=42,
                n_jobs=-1
            )

            model.fit(X_train, y_train)
            y_pred = model.predict(X_val)

            balance_results.append({
                "Setting": setting,
                "Repeat": repeat,
                "Fold": fold_number,
                "Accuracy": accuracy_score(y_val, y_pred),
                "Balanced_Accuracy": balanced_accuracy_score(y_val, y_pred),
                "Macro_F1": f1_score(
                    y_val, y_pred,
                    labels=class_names,
                    average="macro",
                    zero_division=0
                )
            })

            for i, pred in zip(val_idx, y_pred):
                balance_predictions.append({
                    "Setting": setting,
                    "Repeat": repeat,
                    "Fold": fold_number,
                    "Row_Index": int(i),
                    "True_Label": y_array[i],
                    "Predicted_Label": pred
                })

    print(f"Finished {setting}.")

balance_fold_metrics = pd.DataFrame(balance_results)
balance_predictions_df = pd.DataFrame(balance_predictions)

balance_summary = (
    balance_fold_metrics.groupby("Setting")
    .agg(
        Accuracy_Mean=("Accuracy", "mean"),
        Balanced_Accuracy_Mean=("Balanced_Accuracy", "mean"),
        Macro_F1_Mean=("Macro_F1", "mean"),
        Macro_F1_SD=("Macro_F1", "std")
    )
    .reset_index()
)

balance_fold_metrics.to_csv(
    "/kaggle/working/phase2_F3_balance_fold_metrics.csv",
    index=False
)
balance_predictions_df.to_csv(
    "/kaggle/working/phase2_F3_balance_predictions.csv",
    index=False
)
balance_summary.to_csv(
    "/kaggle/working/phase2_F3_balance_summary.csv",
    index=False
)

print("\nF3 BALANCING COMPARISON")
display(balance_summary)



Running F3_no_oversampling...


/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")


Finished F3_no_oversampling.

Running F3_random_oversampling...


/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")


Finished F3_random_oversampling.

F3 BALANCING COMPARISON


,Setting,Accuracy_Mean,Balanced_Accuracy_Mean,Macro_F1_Mean,Macro_F1_SD
0,F3_no_oversampling,0.900351,0.432132,0.395877,0.094875
1,F3_random_oversampling,0.889086,0.434384,0.393372,0.096624


In [47]:

from sklearn.metrics import classification_report

class_names = ["N", "S", "V", "F", "Q"]

for setting in [
    "F3_no_oversampling",
    "F3_random_oversampling"
]:
    print(f"\n{'=' * 60}")
    print(setting)
    print("=" * 60)

    subset = balance_predictions_df[
        balance_predictions_df["Setting"] == setting
    ]

    print(classification_report(
        subset["True_Label"],
        subset["Predicted_Label"],
        labels=class_names,
        target_names=class_names,
        zero_division=0,
        digits=4
    ))



F3_no_oversampling
              precision    recall  f1-score   support

           N     0.9438    0.9445    0.9441    451705
           S     0.0093    0.0014    0.0025     13905
           V     0.5807    0.8087    0.6760     36145
           F     0.0024    0.0022    0.0023      4010
           Q     0.9743    0.8477    0.9066     19470

    accuracy                         0.8994    525235
   macro avg     0.5021    0.5209    0.5063    525235
weighted avg     0.8880    0.8994    0.8922    525235


F3_random_oversampling
              precision    recall  f1-score   support

           N     0.9446    0.9292    0.9369    451705
           S     0.0080    0.0014    0.0023     13905
           V     0.5530    0.8084    0.6567     36145
           F     0.0013    0.0027    0.0018      4010
           Q     0.9884    0.8822    0.9323     19470

    accuracy                         0.8875    525235
   macro avg     0.4991    0.5248    0.5060    525235
weighted avg     0.8873    0.8875

In [49]:

import pickle
import os
from datetime import datetime

phase2_checkpoint_v3 = {
    "checkpoint_name": "MIT-BIH Phase 2 - Balancing Results",
    "created_at": datetime.now().isoformat(),
    "objects": {}
}

variables_to_save = [
    "aami_df",
    "split_indices",
    "mapping",
    "DERIVATIVE_FEATURES",
    "X_F1",
    "X_F2",
    "X_F3",
    "y_phase2",
    "phase2_fold_metrics",
    "phase2_predictions_df",
    "phase2_summary",
    "norm_fold_metrics",
    "norm_predictions_df",
    "norm_summary",
    "balance_fold_metrics",
    "balance_predictions_df",
    "balance_summary",
]

for name in variables_to_save:
    if name in globals():
        phase2_checkpoint_v3["objects"][name] = globals()[name]

checkpoint_path = "/kaggle/working/phase2_checkpoint_v3.pkl"

with open(checkpoint_path, "wb") as f:
    pickle.dump(
        phase2_checkpoint_v3,
        f,
        protocol=pickle.HIGHEST_PROTOCOL
    )

print("Checkpoint saved:", checkpoint_path)
print("Size (MB):", round(os.path.getsize(checkpoint_path) / (1024**2), 2))
print("Saved variables:", list(phase2_checkpoint_v3["objects"].keys()))


Checkpoint saved: /kaggle/working/phase2_checkpoint_v3.pkl
Size (MB): 491.72
Saved variables: ['aami_df', 'split_indices', 'mapping', 'DERIVATIVE_FEATURES', 'X_F1', 'X_F2', 'X_F3', 'y_phase2', 'phase2_fold_metrics', 'phase2_predictions_df', 'phase2_summary', 'norm_fold_metrics', 'norm_predictions_df', 'norm_summary', 'balance_fold_metrics', 'balance_predictions_df', 'balance_summary']


In [50]:

import numpy as np
import pandas as pd

# Pivot so each row contains paired scores from the same fold
paired = norm_fold_metrics.pivot(
    index=["Repeat", "Fold"],
    columns="Feature_Set",
    values="Macro_F1"
).dropna()

paired["Difference_F3_minus_F2"] = (
    paired["F3_MLII_normalized"] - paired["F2_MLII_original"]
)

print("Number of paired fold scores:", len(paired))
print("\nPaired Macro-F1 difference summary:")
print(paired["Difference_F3_minus_F2"].describe())

print(
    "\nF3 wins on",
    int((paired["Difference_F3_minus_F2"] > 0).sum()),
    "of",
    len(paired),
    "folds."
)

print(
    "F2 wins on",
    int((paired["Difference_F3_minus_F2"] < 0).sum()),
    "folds."
)

print(
    "Ties:",
    int((paired["Difference_F3_minus_F2"] == 0).sum())
)


Number of paired fold scores: 25

Paired Macro-F1 difference summary:
count    25.000000
mean      0.053252
std       0.045381
min      -0.007255
25%       0.023350
50%       0.035005
75%       0.093670
max       0.140512
Name: Difference_F3_minus_F2, dtype: float64

F3 wins on 24 of 25 folds.
F2 wins on 1 folds.
Ties: 0


In [51]:

# Save paired F2 vs F3 comparison
paired.to_csv(
    "/kaggle/working/phase2_F2_vs_F3_paired_comparison.csv",
    index=True
)

paired_summary = pd.DataFrame([{
    "Comparison": "F3_normalized_minus_F2_original",
    "Number_of_paired_folds": len(paired),
    "Mean_difference": paired["Difference_F3_minus_F2"].mean(),
    "Median_difference": paired["Difference_F3_minus_F2"].median(),
    "SD_difference": paired["Difference_F3_minus_F2"].std(),
    "F3_wins": int((paired["Difference_F3_minus_F2"] > 0).sum()),
    "F2_wins": int((paired["Difference_F3_minus_F2"] < 0).sum()),
    "Ties": int((paired["Difference_F3_minus_F2"] == 0).sum())
}])

paired_summary.to_csv(
    "/kaggle/working/phase2_F2_vs_F3_paired_summary.csv",
    index=False
)

display(paired_summary)
print("Paired comparison files saved.")


,Comparison,Number_of_paired_folds,Mean_difference,Median_difference,SD_difference,F3_wins,F2_wins,Ties
0,F3_normalized_minus_F2_original,25,0.053252,0.035005,0.045381,24,1,0


Paired comparison files saved.


In [52]:

# ============================================================
# PHASE 2 — F4: Normalized MLII derivatives + waveform shape
# ============================================================

import numpy as np
import pandas as pd

# Reuse normalized_waveforms created earlier.
# If the variable is missing, recreate it from the ECG windows.
if "normalized_waveforms" not in globals():
    mlii_cols = [f"MLII_{i}" for i in range(201)]
    waveforms = aami_df[mlii_cols].to_numpy(dtype=np.float32)

    means = waveforms.mean(axis=1, keepdims=True)
    stds = waveforms.std(axis=1, keepdims=True)
    stds[stds == 0] = 1.0

    normalized_waveforms = (waveforms - means) / stds

# Distribution-based waveform-shape features
q25 = np.percentile(normalized_waveforms, 25, axis=1)
q50 = np.percentile(normalized_waveforms, 50, axis=1)
q75 = np.percentile(normalized_waveforms, 75, axis=1)

X_F4_shape = pd.DataFrame({
    "MLII_norm_median": q50,
    "MLII_norm_IQR": q75 - q25,
    "MLII_norm_q25": q25,
    "MLII_norm_q75": q75,
}, index=aami_df.index)

# Combine with the existing normalized derivative features
X_F4 = pd.concat([X_F3, X_F4_shape], axis=1)

assert X_F4.index.equals(aami_df.index)
assert X_F4.shape == (len(aami_df), 12)
assert np.isfinite(X_F4.to_numpy()).all()

print("F3 normalized derivatives:", X_F3.shape)
print("F4 derivatives + waveform shape:", X_F4.shape)
print("\nNew shape features:")
display(X_F4_shape.head())

print("PASS: F4 features are aligned and contain finite values.")


F3 normalized derivatives: (105047, 8)
F4 derivatives + waveform shape: (105047, 12)

New shape features:


,MLII_norm_median,MLII_norm_IQR,MLII_norm_q25,MLII_norm_q75
0,-0.294554,0.471846,-0.429367,0.042479
1,-0.280286,0.415865,-0.402599,0.013266
2,-0.212347,0.258364,-0.289857,-0.031493
3,-0.274175,0.410241,-0.383572,0.026668
4,-0.315781,0.463863,-0.418861,0.045002


PASS: F4 features are aligned and contain finite values.


In [53]:

# ============================================================
# PHASE 2 — Compare F3 vs F4
# ============================================================

import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score
)

y_array = aami_df["AAMI_Label"].to_numpy()

feature_sets_f4 = {
    "F3_normalized_derivatives": X_F3,
    "F4_derivatives_plus_shape": X_F4,
}

f4_fold_results = []
f4_predictions = []

for feature_name, X_features in feature_sets_f4.items():
    print(f"\nRunning {feature_name}...")
    X_array = X_features.to_numpy()

    for repeat, folds in split_indices.items():
        for fold_number, (train_idx, val_idx) in enumerate(folds, start=1):
            model = RandomForestClassifier(
                n_estimators=200,
                max_features="log2",
                min_samples_split=2,
                min_samples_leaf=1,
                random_state=42,
                n_jobs=-1
            )

            model.fit(X_array[train_idx], y_array[train_idx])
            y_pred = model.predict(X_array[val_idx])
            y_true = y_array[val_idx]

            f4_fold_results.append({
                "Feature_Set": feature_name,
                "Repeat": repeat,
                "Fold": fold_number,
                "Accuracy": accuracy_score(y_true, y_pred),
                "Balanced_Accuracy": balanced_accuracy_score(
                    y_true, y_pred
                ),
                "Macro_F1": f1_score(
                    y_true, y_pred,
                    labels=["N", "S", "V", "F", "Q"],
                    average="macro",
                    zero_division=0
                )
            })

            for i, pred in zip(val_idx, y_pred):
                f4_predictions.append({
                    "Feature_Set": feature_name,
                    "Repeat": repeat,
                    "Fold": fold_number,
                    "Row_Index": int(i),
                    "True_Label": y_array[i],
                    "Predicted_Label": pred
                })

    print(f"Finished {feature_name}.")

f4_fold_metrics = pd.DataFrame(f4_fold_results)
f4_predictions_df = pd.DataFrame(f4_predictions)

f4_summary = (
    f4_fold_metrics.groupby("Feature_Set")
    .agg(
        Accuracy_Mean=("Accuracy", "mean"),
        Accuracy_SD=("Accuracy", "std"),
        Balanced_Accuracy_Mean=("Balanced_Accuracy", "mean"),
        Balanced_Accuracy_SD=("Balanced_Accuracy", "std"),
        Macro_F1_Mean=("Macro_F1", "mean"),
        Macro_F1_SD=("Macro_F1", "std")
    )
    .reset_index()
)

f4_fold_metrics.to_csv(
    "/kaggle/working/phase2_F3_F4_fold_metrics.csv",
    index=False
)
f4_predictions_df.to_csv(
    "/kaggle/working/phase2_F3_F4_predictions.csv",
    index=False
)
f4_summary.to_csv(
    "/kaggle/working/phase2_F3_F4_summary.csv",
    index=False
)

print("\nF3 VS F4 SUMMARY")
display(f4_summary)
print("\nSaved the fold metrics, predictions, and summary.")



Running F3_normalized_derivatives...


/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")


Finished F3_normalized_derivatives.

Running F4_derivatives_plus_shape...


/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")


Finished F4_derivatives_plus_shape.

F3 VS F4 SUMMARY


,Feature_Set,Accuracy_Mean,Accuracy_SD,Balanced_Accuracy_Mean,Balanced_Accuracy_SD,Macro_F1_Mean,Macro_F1_SD
0,F3_normalized_derivatives,0.900351,0.045870,0.432132,0.088040,0.395877,0.094875
1,F4_derivatives_plus_shape,0.909523,0.054318,0.457877,0.090022,0.411359,0.095183



Saved the fold metrics, predictions, and summary.


In [54]:

import pandas as pd
import numpy as np

# Keep only F3 and F4 results
paired = f4_fold_metrics[
    f4_fold_metrics["Feature_Set"].isin([
        "F3_normalized_derivatives",
        "F4_derivatives_plus_shape"
    ])
].copy()

# Check that both feature sets have the same folds
fold_cols = ["Repeat", "Fold"]
required_cols = fold_cols + ["Feature_Set", "Macro_F1"]

missing = [c for c in required_cols if c not in paired.columns]
if missing:
    raise ValueError(
        f"Missing columns: {missing}. "
        f"Available columns: {paired.columns.tolist()}"
    )

wide = paired.pivot(
    index=fold_cols,
    columns="Feature_Set",
    values="Macro_F1"
).reset_index()

f3_col = "F3_normalized_derivatives"
f4_col = "F4_derivatives_plus_shape"

if wide[[f3_col, f4_col]].isna().any().any():
    raise ValueError("Some folds are missing an F3 or F4 score.")

wide["Macro_F1_Difference_F4_minus_F3"] = (
    wide[f4_col] - wide[f3_col]
)

print("PAIRED F3 VS F4 MACRO-F1 COMPARISON")
print("Number of matched folds:", len(wide))
print("Mean F3 Macro-F1:", round(wide[f3_col].mean(), 6))
print("Mean F4 Macro-F1:", round(wide[f4_col].mean(), 6))
print(
    "Mean paired difference:",
    round(wide["Macro_F1_Difference_F4_minus_F3"].mean(), 6)
)
print(
    "Median paired difference:",
    round(wide["Macro_F1_Difference_F4_minus_F3"].median(), 6)
)
print(
    "F4 wins:",
    int((wide["Macro_F1_Difference_F4_minus_F3"] > 0).sum()),
    "of", len(wide), "folds"
)
print(
    "F3 wins:",
    int((wide["Macro_F1_Difference_F4_minus_F3"] < 0).sum()),
    "of", len(wide), "folds"
)
print(
    "Ties:",
    int((wide["Macro_F1_Difference_F4_minus_F3"] == 0).sum())
)

output_path = "/kaggle/working/phase2_F3_F4_paired_comparison.csv"
wide.to_csv(output_path, index=False)
print("\nSaved:", output_path)

display(wide.head())


PAIRED F3 VS F4 MACRO-F1 COMPARISON
Number of matched folds: 25
Mean F3 Macro-F1: 0.395877
Mean F4 Macro-F1: 0.411359
Mean paired difference: 0.015482
Median paired difference: 0.013599
F4 wins: 20 of 25 folds
F3 wins: 5 of 25 folds
Ties: 0

Saved: /kaggle/working/phase2_F3_F4_paired_comparison.csv


Feature_Set,Repeat,Fold,F3_normalized_derivatives,F4_derivatives_plus_shape,Macro_F1_Difference_F4_minus_F3
0,1,1,0.386394,0.386282,-0.000112
1,1,2,0.346765,0.358571,0.011806
2,1,3,0.538317,0.559145,0.020827
3,1,4,0.246235,0.239242,-0.006993
4,1,5,0.463761,0.504475,0.040714


In [56]:

import os
import pickle

checkpoint_path = "/kaggle/working/phase2_checkpoint_v4.pkl"

checkpoint = {
    "aami_df": aami_df,
    "split_indices": split_indices,
    "mapping": mapping,
    "DERIVATIVE_FEATURES": DERIVATIVE_FEATURES,
    "X_F3": X_F3,
    "X_F4": X_F4,
    "y": y,
    "f4_fold_metrics": f4_fold_metrics,
    "f4_predictions_df": f4_predictions_df,
    "f4_summary": f4_summary,
    "f3_f4_paired_comparison": wide
}

with open(checkpoint_path, "wb") as f:
    pickle.dump(checkpoint, f, protocol=pickle.HIGHEST_PROTOCOL)

print("Checkpoint saved:", checkpoint_path)
print("File size (MB):", round(os.path.getsize(checkpoint_path) / (1024**2), 2))
print("Saved objects:", list(checkpoint.keys()))


Checkpoint saved: /kaggle/working/phase2_checkpoint_v4.pkl
File size (MB): 398.14
Saved objects: ['aami_df', 'split_indices', 'mapping', 'DERIVATIVE_FEATURES', 'X_F3', 'X_F4', 'y', 'f4_fold_metrics', 'f4_predictions_df', 'f4_summary', 'f3_f4_paired_comparison']


In [57]:
print("Prediction columns:")
print(f4_predictions_df.columns.tolist())

print("\nFirst five rows:")
display(f4_predictions_df.head())

Prediction columns:
['Feature_Set', 'Repeat', 'Fold', 'Row_Index', 'True_Label', 'Predicted_Label']

First five rows:


,Feature_Set,Repeat,Fold,Row_Index,True_Label,Predicted_Label
0,F3_normalized_derivatives,1,1,0,N,N
1,F3_normalized_derivatives,1,1,1,N,N
2,F3_normalized_derivatives,1,1,2,N,N
3,F3_normalized_derivatives,1,1,3,N,N
4,F3_normalized_derivatives,1,1,4,N,N


In [58]:

import pandas as pd
from sklearn.metrics import classification_report, confusion_matrix

# Keep only F4 predictions
f4_preds = f4_predictions_df[
    f4_predictions_df["Feature_Set"] == "F4_derivatives_plus_shape"
].copy()

labels = ["N", "S", "V", "F", "Q"]

print("F4 prediction rows:", len(f4_preds))
print("Unique beats represented:", f4_preds["Row_Index"].nunique())
print("Repeats:", sorted(f4_preds["Repeat"].unique()))

# Pooled report across repeated predictions
print("\nF4 CLASSIFICATION REPORT")
print(classification_report(
    f4_preds["True_Label"],
    f4_preds["Predicted_Label"],
    labels=labels,
    target_names=[
        "N - Normal",
        "S - Supraventricular ectopic",
        "V - Ventricular ectopic",
        "F - Fusion",
        "Q - Unknown/paced"
    ],
    zero_division=0,
    digits=4
))

# Confusion matrix
cm = confusion_matrix(
    f4_preds["True_Label"],
    f4_preds["Predicted_Label"],
    labels=labels
)

cm_df = pd.DataFrame(
    cm,
    index=[f"Actual_{x}" for x in labels],
    columns=[f"Predicted_{x}" for x in labels]
)

print("\nF4 CONFUSION MATRIX")
display(cm_df)

# Save results
report_dict = classification_report(
    f4_preds["True_Label"],
    f4_preds["Predicted_Label"],
    labels=labels,
    target_names=labels,
    output_dict=True,
    zero_division=0
)

report_df = pd.DataFrame(report_dict).transpose()
report_df.to_csv("/kaggle/working/phase2_F4_classification_report.csv")
cm_df.to_csv("/kaggle/working/phase2_F4_confusion_matrix.csv")

print("\nSaved:")
print("/kaggle/working/phase2_F4_classification_report.csv")
print("/kaggle/working/phase2_F4_confusion_matrix.csv")


F4 prediction rows: 525235
Unique beats represented: 105047
Repeats: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5)]

F4 CLASSIFICATION REPORT
                              precision    recall  f1-score   support

                  N - Normal     0.9491    0.9503    0.9497    451705
S - Supraventricular ectopic     0.0130    0.0006    0.0011     13905
     V - Ventricular ectopic     0.5851    0.8487    0.6927     36145
                  F - Fusion     0.0052    0.0030    0.0038      4010
           Q - Unknown/paced     0.9925    0.8986    0.9432     19470

                    accuracy                         0.9090    525235
                   macro avg     0.5090    0.5402    0.5181    525235
                weighted avg     0.8937    0.9090    0.8994    525235


F4 CONFUSION MATRIX


,Predicted_N,Predicted_S,Predicted_V,Predicted_F,Predicted_Q
Actual_N,429249,590,19647,2098,121
Actual_S,13111,8,731,53,2
Actual_V,5297,16,30678,145,9
Actual_F,2772,0,1226,12,0
Actual_Q,1823,1,151,0,17495



Saved:
/kaggle/working/phase2_F4_classification_report.csv
/kaggle/working/phase2_F4_confusion_matrix.csv


In [61]:
print("Split dictionary keys:", list(split_indices.keys()))

for key in list(split_indices.keys())[:2]:
    value = split_indices[key]
    print(f"\nKey: {key}")
    print("Value type:", type(value))
    print("Value length:", len(value) if hasattr(value, "__len__") else "N/A")
    print("Value preview:", repr(value)[:500])

Split dictionary keys: [1, 2, 3, 4, 5]

Key: 1
Value type: <class 'list'>
Value length: 5
Value preview: [(array([  2271,   2272,   2273, ..., 105044, 105045, 105046]), array([     0,      1,      2, ..., 102291, 102292, 102293])), (array([     0,      1,      2, ..., 102291, 102292, 102293]), array([ 19371,  19372,  19373, ..., 105044, 105045, 105046])), (array([     0,      1,      2, ..., 105044, 105045, 105046]), array([ 2271,  2272,  2273, ..., 91555, 91556, 91557])), (array([     0,      1,      2, ..., 105044, 105045, 105046]), array([17247, 17248, 17249, ..., 99214, 99215, 99216])), (array(

Key: 2
Value type: <class 'list'>
Value length: 5
Value preview: [(array([  4135,   4136,   4137, ..., 105044, 105045, 105046]), array([    0,     1,     2, ..., 99214, 99215, 99216])), (array([     0,      1,      2, ..., 105044, 105045, 105046]), array([  8791,   8792,   8793, ..., 102291, 102292, 102293])), (array([     0,      1,      2, ..., 102291, 102292, 102293]), array([ 17247,  17

In [62]:

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score

classes = ["N", "S", "V", "F", "Q"]
f5_fold_rows = []
f5_prediction_rows = []

for repeat_idx, repeat_splits in split_indices.items():
    for fold_idx, (train_idx, val_idx) in enumerate(repeat_splits, start=1):

        X_train = X_F4.iloc[train_idx]
        X_val = X_F4.iloc[val_idx]

        y_train = y.iloc[train_idx] if hasattr(y, "iloc") else y[train_idx]
        y_val = y.iloc[val_idx] if hasattr(y, "iloc") else y[val_idx]

        model = RandomForestClassifier(
            n_estimators=200,
            max_features="log2",
            min_samples_split=2,
            min_samples_leaf=1,
            class_weight="balanced",
            random_state=42,
            n_jobs=-1
        )

        model.fit(X_train, y_train)
        y_pred = model.predict(X_val)

        f5_fold_rows.append({
            "Feature_Set": "F5_F4_class_weight_balanced",
            "Repeat": repeat_idx,
            "Fold": fold_idx,
            "Accuracy": accuracy_score(y_val, y_pred),
            "Balanced_Accuracy": balanced_accuracy_score(y_val, y_pred),
            "Macro_F1": f1_score(
                y_val, y_pred,
                labels=classes,
                average="macro",
                zero_division=0
            )
        })

        f5_prediction_rows.append(pd.DataFrame({
            "Feature_Set": "F5_F4_class_weight_balanced",
            "Repeat": repeat_idx,
            "Fold": fold_idx,
            "Row_Index": np.asarray(val_idx),
            "True_Label": np.asarray(y_val),
            "Predicted_Label": np.asarray(y_pred)
        }))

    print(f"Finished repeat {repeat_idx}/5")

f5_fold_metrics = pd.DataFrame(f5_fold_rows)
f5_predictions_df = pd.concat(f5_prediction_rows, ignore_index=True)

f5_summary = (
    f5_fold_metrics[["Accuracy", "Balanced_Accuracy", "Macro_F1"]]
    .agg(["mean", "std"])
    .T.rename(columns={"mean": "Mean", "std": "SD"})
)

print("\nF5 CLASS-WEIGHTED RESULTS")
display(f5_summary)

f5_fold_metrics.to_csv(
    "/kaggle/working/phase2_F5_fold_metrics.csv", index=False
)
f5_predictions_df.to_csv(
    "/kaggle/working/phase2_F5_predictions.csv", index=False
)
f5_summary.to_csv(
    "/kaggle/working/phase2_F5_summary.csv"
)

print("\nSaved F5 fold metrics, predictions, and summary.")
print("Fold count:", len(f5_fold_metrics))
print("Prediction rows:", len(f5_predictions_df))


Finished repeat 1/5


/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")


Finished repeat 2/5
Finished repeat 3/5


/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")


Finished repeat 4/5
Finished repeat 5/5

F5 CLASS-WEIGHTED RESULTS


,Mean,SD
Accuracy,0.914681,0.048689
Balanced_Accuracy,0.433791,0.082224
Macro_F1,0.407025,0.096511



Saved F5 fold metrics, predictions, and summary.
Fold count: 25
Prediction rows: 525235


In [63]:

from sklearn.metrics import classification_report
import pandas as pd

f4_preds = f4_predictions_df[
    f4_predictions_df["Feature_Set"] == "F4_derivatives_plus_shape"
].copy()

f5_preds = f5_predictions_df.copy()

labels = ["N", "S", "V", "F", "Q"]

for name, preds in [
    ("F4 baseline", f4_preds),
    ("F5 class-weighted", f5_preds)
]:
    print(f"\n{'=' * 20} {name} {'=' * 20}")
    print(classification_report(
        preds["True_Label"],
        preds["Predicted_Label"],
        labels=labels,
        target_names=[
            "N - Normal",
            "S - Supraventricular ectopic",
            "V - Ventricular ectopic",
            "F - Fusion",
            "Q - Unknown/paced"
        ],
        zero_division=0,
        digits=4
    ))



==================== F4 baseline ====================
                              precision    recall  f1-score   support

                  N - Normal     0.9491    0.9503    0.9497    451705
S - Supraventricular ectopic     0.0130    0.0006    0.0011     13905
     V - Ventricular ectopic     0.5851    0.8487    0.6927     36145
                  F - Fusion     0.0052    0.0030    0.0038      4010
           Q - Unknown/paced     0.9925    0.8986    0.9432     19470

                    accuracy                         0.9090    525235
                   macro avg     0.5090    0.5402    0.5181    525235
                weighted avg     0.8937    0.9090    0.8994    525235


==================== F5 class-weighted ====================
                              precision    recall  f1-score   support

                  N - Normal     0.9438    0.9615    0.9526    451705
S - Supraventricular ectopic     0.0055    0.0001    0.0001     13905
     V - Ventricular ectopic     0.6541 

In [64]:

print("F4 prediction columns:")
print(f4_predictions_df.columns.tolist())

print("\nAvailable variables related to F4 or probabilities:")
for name in [
    "f4_models",
    "f4_probabilities",
    "f4_prediction_probabilities",
    "all_predictions",
    "phase2_predictions_df",
    "X_F4",
    "y",
    "split_indices"
]:
    print(f"{name}: {'available' if name in globals() else 'not available'}")


F4 prediction columns:
['Feature_Set', 'Repeat', 'Fold', 'Row_Index', 'True_Label', 'Predicted_Label']

Available variables related to F4 or probabilities:
f4_models: not available
f4_probabilities: not available
f4_prediction_probabilities: not available
all_predictions: available
phase2_predictions_df: available
X_F4: available
y: available
split_indices: available


In [65]:

print("Type of all_predictions:", type(all_predictions))

if isinstance(all_predictions, dict):
    print("Dictionary keys:", list(all_predictions.keys())[:20])
    for key in list(all_predictions.keys())[:3]:
        value = all_predictions[key]
        print(f"\nKey: {key}")
        print("Value type:", type(value))
        if hasattr(value, "columns"):
            print("Columns:", value.columns.tolist())
            display(value.head(2))
        else:
            print("Preview:", repr(value)[:300])

elif isinstance(all_predictions, pd.DataFrame):
    print("Columns:", all_predictions.columns.tolist())
    display(all_predictions.head())
    
else:
    print("Preview:", repr(all_predictions)[:1000])


Type of all_predictions: <class 'pandas.core.frame.DataFrame'>
Columns: ['Run', 'Repeat', 'Fold', 'Row_Index', 'Record_ID', 'Person_ID', 'True_Label', 'Predicted_Label', 'Prob_N', 'Prob_S', 'Prob_V', 'Prob_F', 'Prob_Q']


,Run,Repeat,Fold,Row_Index,Record_ID,Person_ID,True_Label,Predicted_Label,Prob_N,Prob_S,Prob_V,Prob_F,Prob_Q
0,P1_raw_RF100,1,1,0,100,100,N,N,0.85,0.12,0.00,0.03,0.0
1,P1_raw_RF100,1,1,1,100,100,N,N,0.89,0.10,0.00,0.01,0.0
2,P1_raw_RF100,1,1,2,100,100,N,N,0.93,0.07,0.00,0.00,0.0
3,P1_raw_RF100,1,1,3,100,100,N,N,0.88,0.09,0.01,0.02,0.0
4,P1_raw_RF100,1,1,4,100,100,N,N,0.83,0.13,0.01,0.03,0.0


In [66]:

print("Saved model runs:")
print(sorted(all_predictions["Run"].dropna().unique().tolist()))

print("\nRows per run:")
display(
    all_predictions.groupby("Run").size().rename("Prediction_Rows").to_frame()
)


Saved model runs:
['P1_raw_RF100', 'P2_derivative_ANOVA8_RF200', 'P3_derivative_ANOVA8_ROS_RF200']

Rows per run:


,Prediction_Rows
Run,
P1_raw_RF100,525235
P2_derivative_ANOVA8_RF200,525235
P3_derivative_ANOVA8_ROS_RF200,525235


In [70]:

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier

classes = ["N", "S", "V", "F", "Q"]
prob_cols = [f"Prob_{c}" for c in classes]
f4_probability_rows = []

for repeat_idx, repeat_splits in split_indices.items():
    for fold_idx, (train_idx, val_idx) in enumerate(repeat_splits, start=1):

        X_train = X_F4.iloc[train_idx]
        X_val = X_F4.iloc[val_idx]
        y_train = y.iloc[train_idx] if hasattr(y, "iloc") else y[train_idx]

        model = RandomForestClassifier(
            n_estimators=200,
            max_features="log2",
            min_samples_split=2,
            min_samples_leaf=1,
            random_state=42,
            n_jobs=-1
        )
        model.fit(X_train, y_train)

        fold_probs = model.predict_proba(X_val)
        fold_classes = list(model.classes_)

        # Arrange probability columns consistently as N, S, V, F, Q
        aligned_probs = np.zeros((len(val_idx), len(classes)))
        for j, cls in enumerate(classes):
            if cls in fold_classes:
                aligned_probs[:, j] = fold_probs[:, fold_classes.index(cls)]

        out = pd.DataFrame({
            "Repeat": repeat_idx,
            "Fold": fold_idx,
            "Row_Index": np.asarray(val_idx),
            "True_Label": (
                np.asarray(y.iloc[val_idx])
                if hasattr(y, "iloc") else np.asarray(y)[val_idx]
            )
        })

        for j, col in enumerate(prob_cols):
            out[col] = aligned_probs[:, j]

        f4_probability_rows.append(out)

    print(f"Finished repeat {repeat_idx}/5")

f4_probability_df = pd.concat(
    f4_probability_rows, ignore_index=True
)

out_path = "/kaggle/working/phase2_F4_heldout_probabilities.csv"
f4_probability_df.to_csv(out_path, index=False)

print("\nSaved:", out_path)
print("Rows:", len(f4_probability_df))
print("Unique beats:", f4_probability_df["Row_Index"].nunique())
print("Probability columns:", prob_cols)
display(f4_probability_df.head())


KeyboardInterrupt: 

In [71]:
import os

path = "/kaggle/working/phase2_F4_heldout_probabilities.csv"

if os.path.exists(path):
    print("Probability file exists.")
    print("Size (MB):", round(os.path.getsize(path) / (1024**2), 2))
else:
    print("Probability file not saved yet.")

Probability file exists.
Size (MB): 18.49


In [72]:

import os
import pandas as pd

path = "/kaggle/working/phase2_F4_heldout_probabilities.csv"

if os.path.exists(path):
    print("File size (MB):", round(os.path.getsize(path) / (1024**2), 2))

    probs = pd.read_csv(path)

    print("Rows:", len(probs))
    print("Columns:", probs.columns.tolist())

    if "Repeat" in probs.columns and "Fold" in probs.columns:
        print("\nRows by repeat:")
        print(probs.groupby("Repeat").size())

        print("\nFolds per repeat:")
        print(probs.groupby("Repeat")["Fold"].nunique())

    if "Row_Index" in probs.columns:
        print("\nUnique beats:", probs["Row_Index"].nunique())

    required = [
        "Repeat", "Fold", "Row_Index", "True_Label",
        "Prob_N", "Prob_S", "Prob_V", "Prob_F", "Prob_Q"
    ]

    if all(c in probs.columns for c in required):
        print("\nMissing values in required columns:")
        print(probs[required].isna().sum())

        print("\nProbability row sums (first 5):")
        print(probs[[f"Prob_{c}" for c in ["N", "S", "V", "F", "Q"]]]
              .sum(axis=1).head().round(4).tolist())
else:
    print("Probability file not found.")


File size (MB): 18.49
Rows: 525235
Columns: ['Repeat', 'Fold', 'Row_Index', 'True_Label', 'Prob_N', 'Prob_S', 'Prob_V', 'Prob_F', 'Prob_Q']

Rows by repeat:
Repeat
1    105047
2    105047
3    105047
4    105047
5    105047
dtype: int64

Folds per repeat:
Repeat
1    5
2    5
3    5
4    5
5    5
Name: Fold, dtype: int64

Unique beats: 105047

Missing values in required columns:
Repeat        0
Fold          0
Row_Index     0
True_Label    0
Prob_N        0
Prob_S        0
Prob_V        0
Prob_F        0
Prob_Q        0
dtype: int64

Probability row sums (first 5):
[1.0, 1.0, 1.0, 1.0, 1.0]


In [73]:

import numpy as np
import pandas as pd

print("Number of repeats:", len(split_indices))
print("Repeat keys:", list(split_indices.keys()))

# Check the number of folds in every repeat
for repeat_id, folds in split_indices.items():
    print(f"Repeat {repeat_id}: {len(folds)} folds")

# Verify patient separation for every outer fold
patient_ids = np.asarray(person_ids)

overlap_count = 0

for repeat_id, folds in split_indices.items():
    for fold_id, (train_idx, val_idx) in enumerate(folds, start=1):
        train_people = set(patient_ids[train_idx])
        val_people = set(patient_ids[val_idx])
        overlap = train_people.intersection(val_people)

        if overlap:
            overlap_count += 1
            print(
                f"WARNING: patient overlap in repeat {repeat_id}, "
                f"fold {fold_id}: {overlap}"
            )

print("\nPatient-overlap folds:", overlap_count)

if overlap_count == 0:
    print("PASS: no patient overlap in any outer fold.")


Number of repeats: 5
Repeat keys: [1, 2, 3, 4, 5]
Repeat 1: 5 folds
Repeat 2: 5 folds
Repeat 3: 5 folds
Repeat 4: 5 folds
Repeat 5: 5 folds

Patient-overlap folds: 0
PASS: no patient overlap in any outer fold.


In [74]:

import numpy as np
from sklearn.model_selection import GroupShuffleSplit

# Ensure patient IDs are aligned with feature rows
patient_ids = np.asarray(person_ids)
assert len(patient_ids) == len(X_F4), "Patient IDs and F4 features are misaligned."

inner_split_indices = {}

for repeat_id, outer_folds in split_indices.items():
    for fold_id, (outer_train_idx, outer_val_idx) in enumerate(
        outer_folds, start=1
    ):
        # Split outer-training data by patient, not by beat
        inner_splitter = GroupShuffleSplit(
            n_splits=1,
            test_size=0.20,
            random_state=1000 + repeat_id * 10 + fold_id
        )

        inner_train_rel, inner_val_rel = next(
            inner_splitter.split(
                X_F4.iloc[outer_train_idx],
                groups=patient_ids[outer_train_idx]
            )
        )

        inner_train_idx = outer_train_idx[inner_train_rel]
        inner_val_idx = outer_train_idx[inner_val_rel]

        # Verify patient separation across all three partitions
        outer_train_people = set(patient_ids[outer_train_idx])
        outer_val_people = set(patient_ids[outer_val_idx])
        inner_train_people = set(patient_ids[inner_train_idx])
        inner_val_people = set(patient_ids[inner_val_idx])

        assert inner_train_people.isdisjoint(inner_val_people)
        assert inner_train_people.isdisjoint(outer_val_people)
        assert inner_val_people.isdisjoint(outer_val_people)

        inner_split_indices[(repeat_id, fold_id)] = (
            inner_train_idx,
            inner_val_idx
        )

print("Outer folds:", sum(len(v) for v in split_indices.values()))
print("Inner splits created:", len(inner_split_indices))
print("Inner training/validation patient overlap: 0")
print("Outer validation patients kept separate: PASS")

# Show sizes for the first outer fold
train_idx, val_idx = inner_split_indices[(1, 1)]
print("\nExample — repeat 1, fold 1:")
print("Inner training beats:", len(train_idx))
print("Inner validation beats:", len(val_idx))
print("Inner training patients:", len(set(patient_ids[train_idx])))
print("Inner validation patients:", len(set(patient_ids[val_idx])))


Outer folds: 25
Inner splits created: 25
Inner training/validation patient overlap: 0
Outer validation patients kept separate: PASS

Example — repeat 1, fold 1:
Inner training beats: 66268
Inner validation beats: 18365
Inner training patients: 28
Inner validation patients: 8


In [75]:

import numpy as np
from sklearn.ensemble import RandomForestClassifier

repeat_id, fold_id = 1, 1
inner_train_idx, inner_val_idx = inner_split_indices[(repeat_id, fold_id)]

classes = ["N", "S", "V", "F", "Q"]

y_array = np.asarray(y)
X_train = X_F4.iloc[inner_train_idx]
X_val = X_F4.iloc[inner_val_idx]
y_train = y_array[inner_train_idx]
y_val = y_array[inner_val_idx]

model_test = RandomForestClassifier(
    n_estimators=200,
    max_features="log2",
    min_samples_split=2,
    min_samples_leaf=1,
    random_state=42,
    n_jobs=-1
)

print("Training one inner model...")
model_test.fit(X_train, y_train)

raw_probs = model_test.predict_proba(X_val)
model_classes = list(model_test.classes_)

# Align probability columns to the fixed order N, S, V, F, Q
inner_probs = np.zeros((len(inner_val_idx), len(classes)))

for j, cls in enumerate(classes):
    if cls in model_classes:
        inner_probs[:, j] = raw_probs[:, model_classes.index(cls)]

print("\nInner model test complete.")
print("Training beats:", len(inner_train_idx))
print("Validation beats:", len(inner_val_idx))
print("Classes learned:", model_classes)
print("Probability matrix shape:", inner_probs.shape)
print("All probabilities finite:", np.isfinite(inner_probs).all())
print(
    "Probability sums approximately 1:",
    np.allclose(inner_probs.sum(axis=1), 1.0, atol=1e-6)
)
print("\nValidation class counts:")
print({cls: int(np.sum(y_val == cls)) for cls in classes})


Training one inner model...

Inner model test complete.
Training beats: 66268
Validation beats: 18365
Classes learned: ['F', 'N', 'Q', 'S', 'V']
Probability matrix shape: (18365, 5)
All probabilities finite: True
Probability sums approximately 1: True

Validation class counts:
{'N': 13052, 'S': 393, 'V': 1033, 'F': 2, 'Q': 3885}


In [76]:

import numpy as np
import pandas as pd

classes = ["N", "S", "V", "F", "Q"]
y_array = np.asarray(y)

coverage_rows = []

for (repeat_id, fold_id), (train_idx, val_idx) in inner_split_indices.items():
    counts = {
        cls: int(np.sum(y_array[val_idx] == cls))
        for cls in classes
    }

    coverage_rows.append({
        "Repeat": repeat_id,
        "Fold": fold_id,
        "Validation_Beats": len(val_idx),
        **{f"Count_{cls}": counts[cls] for cls in classes}
    })

coverage_df = pd.DataFrame(coverage_rows)

print("CLASS COUNTS ACROSS INNER VALIDATION SPLITS")
display(
    coverage_df[
        ["Count_N", "Count_S", "Count_V", "Count_F", "Count_Q"]
    ].agg(["min", "median", "max"]).T
)

print("\nSplits with fewer than 10 F examples:")
display(
    coverage_df.loc[
        coverage_df["Count_F"] < 10,
        ["Repeat", "Fold", "Count_F"]
    ]
)

print("\nTotal inner validation F examples across splits:",
      int(coverage_df["Count_F"].sum()))


CLASS COUNTS ACROSS INNER VALIDATION SPLITS


,min,median,max
Count_N,12192.0,15959.0,19842.0
Count_S,33.0,260.0,1864.0
Count_V,254.0,1043.0,2326.0
Count_F,1.0,18.0,744.0
Count_Q,0.0,2.0,3885.0



Splits with fewer than 10 F examples:


,Repeat,Fold,Count_F
0,1,1,2
1,1,2,7
2,1,3,6
4,1,5,5
9,2,5,7
11,3,2,1
15,4,1,4
17,4,3,3



Total inner validation F examples across splits: 2906


In [77]:

import pandas as pd
import numpy as np

patient_class_counts = (
    pd.DataFrame({
        "Person_ID": np.asarray(person_ids),
        "Label": np.asarray(y)
    })
    .drop_duplicates()
    .groupby(["Person_ID", "Label"])
    .size()
    .unstack(fill_value=0)
)

# Ensure all five classes are represented as columns
for cls in ["N", "S", "V", "F", "Q"]:
    if cls not in patient_class_counts.columns:
        patient_class_counts[cls] = 0

patient_class_counts = patient_class_counts[["N", "S", "V", "F", "Q"]]

print("Number of patients containing each class:")
print((patient_class_counts > 0).sum().to_string())

print("\nNumber of beats per class across patients:")
print(patient_class_counts.sum().to_string())

print("\nPatients containing F beats:")
display(patient_class_counts.loc[patient_class_counts["F"] > 0, ["F"]])

print("\nPatients containing Q beats:")
display(patient_class_counts.loc[patient_class_counts["Q"] > 0, ["Q"]])


Number of patients containing each class:
Label
N    44
S    31
V    34
F    16
Q     7

Number of beats per class across patients:
Label
N    44
S    31
V    34
F    16
Q     7

Patients containing F beats:


Label,F
Person_ID,
108,1
109,1
114,1
124,1
200,1
203,1
205,1
208,1
210,1



Patients containing Q beats:


Label,Q
Person_ID,
101,1
105,1
107,1
203,1
208,1
214,1
217,1


In [78]:

import pandas as pd
import numpy as np

patient_class_beats = (
    pd.DataFrame({
        "Person_ID": np.asarray(person_ids),
        "Label": np.asarray(y)
    })
    .groupby(["Person_ID", "Label"])
    .size()
    .unstack(fill_value=0)
)

for cls in ["N", "S", "V", "F", "Q"]:
    if cls not in patient_class_beats.columns:
        patient_class_beats[cls] = 0

patient_class_beats = patient_class_beats[["N", "S", "V", "F", "Q"]]

print("Total beats per class:")
print(patient_class_beats.sum().to_string())

print("\nNumber of patients containing each class:")
print((patient_class_beats > 0).sum().to_string())

print("\nF-class beat distribution across patients:")
display(
    patient_class_beats.loc[patient_class_beats["F"] > 0, ["F"]]
    .sort_values("F", ascending=False)
)

print("\nQ-class beat distribution across patients:")
display(
    patient_class_beats.loc[patient_class_beats["Q"] > 0, ["Q"]]
    .sort_values("Q", ascending=False)
)


Total beats per class:
Label
N    90341
S     2781
V     7229
F      802
Q     3894

Number of patients containing each class:
Label
N    44
S    31
V    34
F    16
Q     7

F-class beat distribution across patients:


Label,F
Person_ID,
208,372
213,362
223,14
205,11
233,11
210,10
124,5
114,4
person_201_202,3



Q-class beat distribution across patients:


Label,Q
Person_ID,
107,2077
217,1802
105,5
203,4
101,2
208,2
214,2


In [79]:

import numpy as np
import pandas as pd

# Fixed class order used in the saved F4 probabilities
classes = ["N", "S", "V", "F", "Q"]

# Small, predefined grid to limit overfitting
threshold_grid = [
    {"S_multiplier": 1.0, "F_multiplier": 1.0},
    {"S_multiplier": 1.5, "F_multiplier": 1.0},
    {"S_multiplier": 2.0, "F_multiplier": 1.0},
    {"S_multiplier": 1.0, "F_multiplier": 1.5},
    {"S_multiplier": 1.0, "F_multiplier": 2.0},
    {"S_multiplier": 1.5, "F_multiplier": 1.5},
    {"S_multiplier": 2.0, "F_multiplier": 2.0},
]

threshold_grid_df = pd.DataFrame(threshold_grid)

print("F6 threshold grid:")
display(threshold_grid_df)

# Confirm the outer folds and inner splits are available
assert len(split_indices) == 5
assert len(inner_split_indices) == 25

print("Outer repeats:", len(split_indices))
print("Outer folds:", sum(len(v) for v in split_indices.values()))
print("Inner splits:", len(inner_split_indices))
print("PASS: existing split structures are available.")


F6 threshold grid:


,S_multiplier,F_multiplier
0,1.0,1.0
1,1.5,1.0
2,2.0,1.0
3,1.0,1.5
4,1.0,2.0
5,1.5,1.5
6,2.0,2.0


Outer repeats: 5
Outer folds: 25
Inner splits: 25
PASS: existing split structures are available.


In [80]:

import numpy as np
import pandas as pd
from sklearn.metrics import f1_score, recall_score, precision_score

classes = ["N", "S", "V", "F", "Q"]

# Verify that the previous test-fold probabilities are still available
assert "inner_probs" in globals(), "Please rerun the single inner-model test first."
assert "y_val" in globals(), "Validation labels are not available."

# Baseline predictions from the original probabilities
baseline_pred = np.array(classes)[np.argmax(inner_probs, axis=1)]

results = []

for candidate in threshold_grid:
    adjusted = inner_probs.copy()

    # Adjust only S and F scores
    adjusted[:, classes.index("S")] *= candidate["S_multiplier"]
    adjusted[:, classes.index("F")] *= candidate["F_multiplier"]

    pred = np.array(classes)[np.argmax(adjusted, axis=1)]

    results.append({
        "S_multiplier": candidate["S_multiplier"],
        "F_multiplier": candidate["F_multiplier"],
        "Macro_F1": f1_score(
            y_val, pred, labels=classes,
            average="macro", zero_division=0
        ),
        "S_Recall": recall_score(
            y_val, pred, labels=classes,
            average=None, zero_division=0
        )[classes.index("S")],
        "F_Recall": recall_score(
            y_val, pred, labels=classes,
            average=None, zero_division=0
        )[classes.index("F")],
        "S_Predicted_Count": int(np.sum(pred == "S")),
        "F_Predicted_Count": int(np.sum(pred == "F"))
    })

print("Baseline Macro-F1:", round(
    f1_score(y_val, baseline_pred, labels=classes,
             average="macro", zero_division=0), 4
))

print("\nExploratory threshold-rule check")
display(pd.DataFrame(results))
print("\nReminder: this fold has only 2 true F examples.")


Baseline Macro-F1: 0.2671

Exploratory threshold-rule check


,S_multiplier,F_multiplier,Macro_F1,S_Recall,F_Recall,S_Predicted_Count,F_Predicted_Count
0,1.0,1.0,0.267084,0.000000,0.0,40,0
1,1.5,1.0,0.268086,0.010178,0.0,253,0
2,2.0,1.0,0.267489,0.025445,0.0,572,0
3,1.0,1.5,0.267096,0.000000,0.0,40,2
4,1.0,2.0,0.267022,0.000000,0.0,40,129
5,1.5,1.5,0.268098,0.010178,0.0,253,2
6,2.0,2.0,0.267422,0.025445,0.0,572,129



Reminder: this fold has only 2 true F examples.


In [81]:

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier

classes = ["N", "S", "V", "F", "Q"]
prob_cols = [f"Prob_{c}" for c in classes]
inner_probability_rows = []

y_array = np.asarray(y)

for (repeat_id, fold_id), (train_idx, val_idx) in inner_split_indices.items():
    X_train = X_F4.iloc[train_idx]
    X_val = X_F4.iloc[val_idx]
    y_train = y_array[train_idx]

    model = RandomForestClassifier(
        n_estimators=200,
        max_features="log2",
        min_samples_split=2,
        min_samples_leaf=1,
        random_state=42,
        n_jobs=-1
    )
    model.fit(X_train, y_train)

    raw_probs = model.predict_proba(X_val)
    model_classes = list(model.classes_)

    aligned_probs = np.zeros((len(val_idx), len(classes)))
    for j, cls in enumerate(classes):
        if cls in model_classes:
            aligned_probs[:, j] = raw_probs[:, model_classes.index(cls)]

    out = pd.DataFrame({
        "Repeat": repeat_id,
        "Fold": fold_id,
        "Row_Index": val_idx,
        "True_Label": y_array[val_idx]
    })

    for j, col in enumerate(prob_cols):
        out[col] = aligned_probs[:, j]

    inner_probability_rows.append(out)
    print(f"Finished inner model: repeat {repeat_id}, fold {fold_id}")

inner_probability_df = pd.concat(inner_probability_rows, ignore_index=True)

save_path = "/kaggle/working/phase2_F6_inner_probabilities.csv"
inner_probability_df.to_csv(save_path, index=False)

print("\nInner probability generation complete.")
print("Rows:", len(inner_probability_df))
print("Inner splits:", inner_probability_df[["Repeat", "Fold"]].drop_duplicates().shape[0])
print("Saved:", save_path)


Finished inner model: repeat 1, fold 1
Finished inner model: repeat 1, fold 2
Finished inner model: repeat 1, fold 3
Finished inner model: repeat 1, fold 4
Finished inner model: repeat 1, fold 5
Finished inner model: repeat 2, fold 1
Finished inner model: repeat 2, fold 2
Finished inner model: repeat 2, fold 3
Finished inner model: repeat 2, fold 4
Finished inner model: repeat 2, fold 5
Finished inner model: repeat 3, fold 1
Finished inner model: repeat 3, fold 2
Finished inner model: repeat 3, fold 3
Finished inner model: repeat 3, fold 4
Finished inner model: repeat 3, fold 5
Finished inner model: repeat 4, fold 1
Finished inner model: repeat 4, fold 2
Finished inner model: repeat 4, fold 3
Finished inner model: repeat 4, fold 4
Finished inner model: repeat 4, fold 5
Finished inner model: repeat 5, fold 1
Finished inner model: repeat 5, fold 2
Finished inner model: repeat 5, fold 3
Finished inner model: repeat 5, fold 4
Finished inner model: repeat 5, fold 5

Inner probability genera

In [82]:

import numpy as np
import pandas as pd
from sklearn.metrics import (
    f1_score,
    balanced_accuracy_score,
    precision_score,
    recall_score
)

classes = ["N", "S", "V", "F", "Q"]
prob_cols = [f"Prob_{c}" for c in classes]

threshold_grid = [
    {"S_multiplier": 1.0, "F_multiplier": 1.0},
    {"S_multiplier": 1.5, "F_multiplier": 1.0},
    {"S_multiplier": 2.0, "F_multiplier": 1.0},
    {"S_multiplier": 1.0, "F_multiplier": 1.5},
    {"S_multiplier": 1.0, "F_multiplier": 2.0},
    {"S_multiplier": 1.5, "F_multiplier": 1.5},
    {"S_multiplier": 2.0, "F_multiplier": 2.0},
]

prob_df = inner_probability_df.copy()
result_rows = []

for (repeat_id, fold_id), fold_df in prob_df.groupby(
    ["Repeat", "Fold"], sort=True
):
    y_true = fold_df["True_Label"].to_numpy()
    probs = fold_df[prob_cols].to_numpy()

    for candidate in threshold_grid:
        adjusted = probs.copy()
        adjusted[:, classes.index("S")] *= candidate["S_multiplier"]
        adjusted[:, classes.index("F")] *= candidate["F_multiplier"]

        y_pred = np.array(classes)[np.argmax(adjusted, axis=1)]

        result_rows.append({
            "Repeat": repeat_id,
            "Fold": fold_id,
            "S_multiplier": candidate["S_multiplier"],
            "F_multiplier": candidate["F_multiplier"],
            "Macro_F1": f1_score(
                y_true, y_pred, labels=classes,
                average="macro", zero_division=0
            ),
            "Balanced_Accuracy": balanced_accuracy_score(
                y_true, y_pred
            ),
            "S_Precision": precision_score(
                y_true, y_pred, labels=["S"],
                average="macro", zero_division=0
            ),
            "S_Recall": recall_score(
                y_true, y_pred, labels=["S"],
                average="macro", zero_division=0
            ),
            "F_Precision": precision_score(
                y_true, y_pred, labels=["F"],
                average="macro", zero_division=0
            ),
            "F_Recall": recall_score(
                y_true, y_pred, labels=["F"],
                average="macro", zero_division=0
            ),
            "Predicted_S": int(np.sum(y_pred == "S")),
            "Predicted_F": int(np.sum(y_pred == "F"))
        })

f6_inner_metrics = pd.DataFrame(result_rows)

summary = (
    f6_inner_metrics.groupby(
        ["S_multiplier", "F_multiplier"]
    )
    .agg(
        Macro_F1_Mean=("Macro_F1", "mean"),
        Macro_F1_SD=("Macro_F1", "std"),
        Balanced_Accuracy_Mean=("Balanced_Accuracy", "mean"),
        S_Precision_Mean=("S_Precision", "mean"),
        S_Recall_Mean=("S_Recall", "mean"),
        F_Precision_Mean=("F_Precision", "mean"),
        F_Recall_Mean=("F_Recall", "mean"),
        Predicted_S_Mean=("Predicted_S", "mean"),
        Predicted_F_Mean=("Predicted_F", "mean")
    )
    .reset_index()
    .sort_values("Macro_F1_Mean", ascending=False)
)

print("INNER-VALIDATION THRESHOLD COMPARISON")
display(summary.round(4))

f6_inner_metrics.to_csv(
    "/kaggle/working/phase2_F6_inner_fold_metrics.csv",
    index=False
)
summary.to_csv(
    "/kaggle/working/phase2_F6_inner_threshold_summary.csv",
    index=False
)

print("\nSaved:")
print("/kaggle/working/phase2_F6_inner_fold_metrics.csv")
print("/kaggle/working/phase2_F6_inner_threshold_summary.csv")


/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/usr/local/lib/python3.13/dist-packages/sklearn/metrics

INNER-VALIDATION THRESHOLD COMPARISON


,S_multiplier,F_multiplier,Macro_F1_Mean,Macro_F1_SD,Balanced_Accuracy_Mean,S_Precision_Mean,S_Recall_Mean,F_Precision_Mean,F_Recall_Mean,Predicted_S_Mean,Predicted_F_Mean
4,1.5,1.5,0.3403,0.0576,0.3983,0.0165,0.0109,0.0484,0.0931,126.52,485.72
5,2.0,1.0,0.3402,0.0560,0.3916,0.0313,0.0199,0.0545,0.0449,256.28,264.92
1,1.0,1.5,0.3401,0.0573,0.3979,0.0037,0.0031,0.0484,0.0931,31.64,485.76
3,1.5,1.0,0.3401,0.0558,0.3917,0.0165,0.0109,0.0545,0.0449,126.56,264.92
0,1.0,1.0,0.3399,0.0556,0.3914,0.0037,0.0031,0.0545,0.0449,31.64,264.92
6,2.0,2.0,0.3375,0.0577,0.3986,0.0308,0.0199,0.0735,0.1114,255.76,685.20
2,1.0,2.0,0.3373,0.0572,0.3987,0.0037,0.0031,0.0735,0.1130,31.64,685.52



Saved:
/kaggle/working/phase2_F6_inner_fold_metrics.csv
/kaggle/working/phase2_F6_inner_threshold_summary.csv


In [83]:
import pandas as pd
import numpy as np
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    classification_report
)

# Load the held-out outer-fold probabilities
prob_path = "/kaggle/working/phase2_F4_heldout_probabilities.csv"
outer_probs = pd.read_csv(prob_path)

classes = ["N", "S", "V", "F", "Q"]
prob_cols = [f"Prob_{c}" for c in classes]

# Confirm expected data
assert outer_probs[prob_cols].notna().all().all()
assert outer_probs["True_Label"].notna().all()

print("Rows:", len(outer_probs))
print("Unique beats:", outer_probs["Row_Index"].nunique())
print("Outer folds:", outer_probs[["Repeat", "Fold"]].drop_duplicates().shape[0])

# F4 baseline: original argmax predictions
prob_matrix = outer_probs[prob_cols].to_numpy()
true_labels = outer_probs["True_Label"].to_numpy()

f4_pred = np.array(classes)[np.argmax(prob_matrix, axis=1)]

# F6: fixed multipliers selected using inner validation only
adjusted_probs = prob_matrix.copy()
adjusted_probs[:, classes.index("S")] *= 1.5
adjusted_probs[:, classes.index("F")] *= 1.5

f6_pred = np.array(classes)[np.argmax(adjusted_probs, axis=1)]

outer_probs["F4_Prediction"] = f4_pred
outer_probs["F6_Prediction"] = f6_pred

# Evaluate each outer fold separately
fold_rows = []

for (repeat_id, fold_id), group in outer_probs.groupby(["Repeat", "Fold"]):
    y_true = group["True_Label"].to_numpy()

    row = {
        "Repeat": repeat_id,
        "Fold": fold_id,
    }

    for model_name, pred_col in [
        ("F4", "F4_Prediction"),
        ("F6", "F6_Prediction")
    ]:
        y_pred = group[pred_col].to_numpy()

        row[f"{model_name}_Accuracy"] = accuracy_score(y_true, y_pred)
        row[f"{model_name}_Balanced_Accuracy"] = balanced_accuracy_score(
            y_true, y_pred
        )
        row[f"{model_name}_Macro_F1"] = f1_score(
            y_true, y_pred,
            labels=classes,
            average="macro",
            zero_division=0
        )

    fold_rows.append(row)

fold_metrics = pd.DataFrame(fold_rows)

# Summarize mean and standard deviation across the 25 outer folds
metric_summary = []

for model_name in ["F4", "F6"]:
    for metric in ["Accuracy", "Balanced_Accuracy", "Macro_F1"]:
        values = fold_metrics[f"{model_name}_{metric}"]

        metric_summary.append({
            "Model": model_name,
            "Metric": metric,
            "Mean": values.mean(),
            "SD": values.std(ddof=1)
        })

summary_df = pd.DataFrame(metric_summary)

# Paired fold differences: F6 minus F4
fold_metrics["Macro_F1_Difference_F6_minus_F4"] = (
    fold_metrics["F6_Macro_F1"] - fold_metrics["F4_Macro_F1"]
)

print("\n=== Mean outer-fold metrics ===")
display(summary_df.round(4))

print("\n=== Paired Macro-F1 comparison ===")
print(
    "Mean difference:",
    round(fold_metrics["Macro_F1_Difference_F6_minus_F4"].mean(), 5)
)
print(
    "Median difference:",
    round(fold_metrics["Macro_F1_Difference_F6_minus_F4"].median(), 5)
)
print(
    "F6 wins:",
    int((fold_metrics["Macro_F1_Difference_F6_minus_F4"] > 0).sum()),
    "/",
    len(fold_metrics)
)
print(
    "Ties:",
    int((fold_metrics["Macro_F1_Difference_F6_minus_F4"] == 0).sum())
)

# Pooled classification reports across repeated outer predictions
# Note: each beat appears once per repeat, so these rows are not independent.
print("\n=== F4 pooled classification report ===")
print(classification_report(
    true_labels, f4_pred,
    labels=classes,
    zero_division=0,
    digits=4
))

print("\n=== F6 pooled classification report ===")
print(classification_report(
    true_labels, f6_pred,
    labels=classes,
    zero_division=0,
    digits=4
))

# Save results
fold_metrics.to_csv(
    "/kaggle/working/phase2_F6_outer_fold_metrics.csv", index=False
)

summary_df.to_csv(
    "/kaggle/working/phase2_F6_outer_summary.csv", index=False
)

outer_probs.to_csv(
    "/kaggle/working/phase2_F6_outer_predictions.csv", index=False
)

print("\nSaved:")
print("/kaggle/working/phase2_F6_outer_fold_metrics.csv")
print("/kaggle/working/phase2_F6_outer_summary.csv")
print("/kaggle/working/phase2_F6_outer_predictions.csv")

Rows: 525235
Unique beats: 105047
Outer folds: 25


/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")



=== Mean outer-fold metrics ===


,Model,Metric,Mean,SD
0,F4,Accuracy,0.9096,0.0543
1,F4,Balanced_Accuracy,0.4579,0.0900
2,F4,Macro_F1,0.4113,0.0951
3,F6,Accuracy,0.9021,0.0540
4,F6,Balanced_Accuracy,0.4592,0.0899
5,F6,Macro_F1,0.4101,0.0951



=== Paired Macro-F1 comparison ===
Mean difference: -0.00119
Median difference: -0.00078
F6 wins: 10 / 25
Ties: 0

=== F4 pooled classification report ===
              precision    recall  f1-score   support

           N     0.9491    0.9504    0.9498    451705
           S     0.0130    0.0006    0.0011     13905
           V     0.5849    0.8488    0.6926     36145
           F     0.0053    0.0030    0.0038      4010
           Q     0.9925    0.8978    0.9428     19470

    accuracy                         0.9091    525235
   macro avg     0.5090    0.5401    0.5180    525235
weighted avg     0.8937    0.9091    0.8995    525235


=== F6 pooled classification report ===
              precision    recall  f1-score   support

           N     0.9490    0.9418    0.9454    451705
           S     0.0258    0.0044    0.0075     13905
           V     0.5861    0.8446    0.6920     36145
           F     0.0047    0.0057    0.0052      4010
           Q     0.9925    0.8978    0.9428

In [84]:
import pandas as pd
from sklearn.metrics import confusion_matrix

# Load the saved F4 outer-fold predictions
pred_path = "/kaggle/working/phase2_F6_outer_predictions.csv"
pred_df = pd.read_csv(pred_path)

classes = ["N", "S", "V", "F", "Q"]

# Use the original F4 predictions, not F6-adjusted predictions
cm = confusion_matrix(
    pred_df["True_Label"],
    pred_df["F4_Prediction"],
    labels=classes
)

cm_df = pd.DataFrame(
    cm,
    index=[f"Actual_{c}" for c in classes],
    columns=[f"Predicted_{c}" for c in classes]
)

print("F4 pooled confusion matrix across repeated outer predictions:")
display(cm_df)

# Row-normalized matrix: proportion of each actual class assigned to each prediction
row_totals = cm.sum(axis=1, keepdims=True)
cm_normalized = cm / row_totals

cm_norm_df = pd.DataFrame(
    cm_normalized,
    index=[f"Actual_{c}" for c in classes],
    columns=[f"Predicted_{c}" for c in classes]
)

print("\nRow-normalized confusion matrix:")
display(cm_norm_df.round(4))

F4 pooled confusion matrix across repeated outer predictions:


,Predicted_N,Predicted_S,Predicted_V,Predicted_F,Predicted_Q
Actual_N,429299,590,19648,2047,121
Actual_S,13111,8,731,53,2
Actual_V,5298,16,30679,143,9
Actual_F,2772,0,1226,12,0
Actual_Q,1823,1,165,0,17481



Row-normalized confusion matrix:


,Predicted_N,Predicted_S,Predicted_V,Predicted_F,Predicted_Q
Actual_N,0.9504,0.0013,0.0435,0.0045,0.0003
Actual_S,0.9429,0.0006,0.0526,0.0038,0.0001
Actual_V,0.1466,0.0004,0.8488,0.0040,0.0002
Actual_F,0.6913,0.0000,0.3057,0.0030,0.0000
Actual_Q,0.0936,0.0001,0.0085,0.0000,0.8978


In [85]:
import numpy as np
import pandas as pd

# 1. Check the feature matrix
print("F4 feature matrix shape:", X_F4.shape)
print("Labels shape:", y.shape)
print("Number of unique labels:", len(np.unique(y)))

# 2. Check feature column names
if isinstance(X_F4, pd.DataFrame):
    print("\nF4 feature columns:")
    print(X_F4.columns.tolist())
else:
    print("\nX_F4 is a NumPy array.")
    print("Number of feature columns:", X_F4.shape[1])

# 3. Check that features and labels align
assert X_F4.shape[0] == len(y), "Feature and label row counts do not match."
assert X_F4.shape[1] == 12, "Expected 12 F4 features."
assert len(split_indices) == 5, "Expected 5 repeats of cross-validation."

print("\nFeature and label row counts match.")
print("Number of repeats:", len(split_indices))

# 4. Verify patient separation in every outer fold
for repeat_id, folds in split_indices.items():
    for fold_id, (train_idx, val_idx) in enumerate(folds, start=1):
        train_patients = set(person_ids[train_idx])
        val_patients = set(person_ids[val_idx])

        assert train_patients.isdisjoint(val_patients), (
            f"Patient leakage in repeat {repeat_id}, fold {fold_id}"
        )

print("Patient separation verified in all outer folds.")
print("\nReady to proceed to F7 once these checks pass.")

F4 feature matrix shape: (105047, 12)
Labels shape: (105047,)
Number of unique labels: 5

F4 feature columns:
['MLII_diff_mean', 'MLII_diff_std', 'MLII_diff_max', 'MLII_diff_min', 'MLII_diff_abs_mean', 'MLII_diff_energy', 'MLII_second_diff_std', 'MLII_second_diff_abs_mean', 'MLII_norm_median', 'MLII_norm_IQR', 'MLII_norm_q25', 'MLII_norm_q75']

Feature and label row counts match.
Number of repeats: 5
Patient separation verified in all outer folds.

Ready to proceed to F7 once these checks pass.


In [86]:
import numpy as np
import pandas as pd

from sklearn.ensemble import ExtraTreesClassifier
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score
)

classes = ["N", "S", "V", "F", "Q"]

fold_results = []
prediction_rows = []

for repeat_id, folds in split_indices.items():
    for fold_id, (train_idx, val_idx) in enumerate(folds, start=1):

        X_train = X_F4.iloc[train_idx] if isinstance(X_F4, pd.DataFrame) else X_F4[train_idx]
        X_val = X_F4.iloc[val_idx] if isinstance(X_F4, pd.DataFrame) else X_F4[val_idx]

        y_train = y[train_idx]
        y_val = y[val_idx]

        model = ExtraTreesClassifier(
            n_estimators=200,
            class_weight="balanced",
            random_state=42,
            n_jobs=-1
        )

        model.fit(X_train, y_train)
        y_pred = model.predict(X_val)

        fold_results.append({
            "Repeat": repeat_id,
            "Fold": fold_id,
            "Accuracy": accuracy_score(y_val, y_pred),
            "Balanced_Accuracy": balanced_accuracy_score(y_val, y_pred),
            "Macro_F1": f1_score(
                y_val,
                y_pred,
                labels=classes,
                average="macro",
                zero_division=0
            )
        })

        prediction_rows.append(pd.DataFrame({
            "Repeat": repeat_id,
            "Fold": fold_id,
            "Row_Index": val_idx,
            "True_Label": y_val,
            "F7_Prediction": y_pred
        }))

    print(f"Completed repeat {repeat_id}")

f7_fold_metrics = pd.DataFrame(fold_results)
f7_predictions = pd.concat(prediction_rows, ignore_index=True)

# Summarize metrics across the 25 outer folds
f7_summary = pd.DataFrame({
    "Metric": ["Accuracy", "Balanced_Accuracy", "Macro_F1"],
    "Mean": [
        f7_fold_metrics["Accuracy"].mean(),
        f7_fold_metrics["Balanced_Accuracy"].mean(),
        f7_fold_metrics["Macro_F1"].mean()
    ],
    "SD": [
        f7_fold_metrics["Accuracy"].std(ddof=1),
        f7_fold_metrics["Balanced_Accuracy"].std(ddof=1),
        f7_fold_metrics["Macro_F1"].std(ddof=1)
    ]
})

print("\nF7 mean outer-fold metrics:")
display(f7_summary.round(4))

# Save outputs
f7_fold_metrics.to_csv(
    "/kaggle/working/phase2_F7_outer_fold_metrics.csv",
    index=False
)

f7_predictions.to_csv(
    "/kaggle/working/phase2_F7_outer_predictions.csv",
    index=False
)

f7_summary.to_csv(
    "/kaggle/working/phase2_F7_outer_summary.csv",
    index=False
)

print("Saved F7 fold metrics, predictions, and summary.")

Completed repeat 1


/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")


Completed repeat 2
Completed repeat 3


/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")


Completed repeat 4


/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")


Completed repeat 5

F7 mean outer-fold metrics:


,Metric,Mean,SD
0,Accuracy,0.9184,0.0487
1,Balanced_Accuracy,0.4426,0.0860
2,Macro_F1,0.4098,0.0976


Saved F7 fold metrics, predictions, and summary.


In [87]:
import pandas as pd
import numpy as np

# Load fold-level results
f4 = pd.read_csv(
    "/kaggle/working/phase2_F6_outer_fold_metrics.csv"
)
f7 = pd.read_csv(
    "/kaggle/working/phase2_F7_outer_fold_metrics.csv"
)

# Keep only F4 fold-level metrics and rename columns
f4 = f4[[
    "Repeat", "Fold",
    "F4_Accuracy",
    "F4_Balanced_Accuracy",
    "F4_Macro_F1"
]].copy()

f4 = f4.rename(columns={
    "F4_Accuracy": "F4_Accuracy",
    "F4_Balanced_Accuracy": "F4_Balanced_Accuracy",
    "F4_Macro_F1": "F4_Macro_F1"
})

# Merge on repeat and fold
comparison = f4.merge(
    f7,
    on=["Repeat", "Fold"],
    validate="one_to_one"
)

# Calculate paired differences: F7 minus F4
comparison["Accuracy_Diff"] = (
    comparison["Accuracy"] - comparison["F4_Accuracy"]
)
comparison["Balanced_Accuracy_Diff"] = (
    comparison["Balanced_Accuracy"]
    - comparison["F4_Balanced_Accuracy"]
)
comparison["Macro_F1_Diff"] = (
    comparison["Macro_F1"] - comparison["F4_Macro_F1"]
)

print("Number of matched folds:", len(comparison))

print("\nPaired fold comparison:")
print(
    "Mean Macro-F1 difference (F7 - F4):",
    round(comparison["Macro_F1_Diff"].mean(), 5)
)
print(
    "Median Macro-F1 difference:",
    round(comparison["Macro_F1_Diff"].median(), 5)
)
print(
    "F7 wins on Macro-F1:",
    int((comparison["Macro_F1_Diff"] > 0).sum()),
    "of", len(comparison)
)
print(
    "Ties:",
    int((comparison["Macro_F1_Diff"] == 0).sum())
)

print("\nMean paired differences:")
display(
    comparison[[
        "Accuracy_Diff",
        "Balanced_Accuracy_Diff",
        "Macro_F1_Diff"
    ]].mean().to_frame("Mean difference").round(5)
)

# Save the comparison
comparison.to_csv(
    "/kaggle/working/phase2_F4_F7_paired_comparison.csv",
    index=False
)

print("\nSaved: /kaggle/working/phase2_F4_F7_paired_comparison.csv")

Number of matched folds: 25

Paired fold comparison:
Mean Macro-F1 difference (F7 - F4): -0.00151
Median Macro-F1 difference: -0.00234
F7 wins on Macro-F1: 11 of 25
Ties: 0

Mean paired differences:


,Mean difference
Accuracy_Diff,0.00882
Balanced_Accuracy_Diff,-0.01528
Macro_F1_Diff,-0.00151



Saved: /kaggle/working/phase2_F4_F7_paired_comparison.csv


In [88]:
import pandas as pd
from sklearn.metrics import classification_report

pred_path = "/kaggle/working/phase2_F7_outer_predictions.csv"
pred_df = pd.read_csv(pred_path)

classes = ["N", "S", "V", "F", "Q"]

print("F7 pooled classification report:")
print(
    classification_report(
        pred_df["True_Label"],
        pred_df["F7_Prediction"],
        labels=classes,
        zero_division=0,
        digits=4
    )
)

print("\nF7 predictions per class:")
display(
    pred_df["F7_Prediction"]
    .value_counts()
    .reindex(classes, fill_value=0)
    .rename_axis("Class")
    .to_frame("Predicted_Count")
)

F7 pooled classification report:
              precision    recall  f1-score   support

           N     0.9469    0.9629    0.9548    451705
           S     0.0000    0.0000    0.0000     13905
           V     0.6432    0.8286    0.7242     36145
           F     0.0026    0.0012    0.0017      4010
           Q     0.9980    0.8854    0.9383     19470

    accuracy                         0.9180    525235
   macro avg     0.5181    0.5356    0.5238    525235
weighted avg     0.8956    0.9180    0.9058    525235


F7 predictions per class:


,Predicted_Count
Class,
N,459336
S,132
V,46564
F,1931
Q,17272


In [89]:
import pandas as pd
import numpy as np

# Load F4 held-out outer-fold probabilities
path = "/kaggle/working/phase2_F4_heldout_probabilities.csv"
probs_df = pd.read_csv(path)

classes = ["N", "S", "V", "F", "Q"]

# Check probability columns
prob_cols = [f"Prob_{c}" for c in classes]
assert all(c in probs_df.columns for c in prob_cols)

print("Rows:", len(probs_df))
print("Unique beats:", probs_df["Row_Index"].nunique())

# For each actual class, summarize the probability assigned
# to S and F by the model.
rows = []

for actual_class in classes:
    subset = probs_df[probs_df["True_Label"] == actual_class]

    for target_class in ["S", "F"]:
        values = subset[f"Prob_{target_class}"]

        rows.append({
            "Actual_Class": actual_class,
            "Probability_Target": target_class,
            "Count": len(values),
            "Mean_Probability": values.mean(),
            "Median_Probability": values.median(),
            "90th_Percentile": values.quantile(0.90),
            "99th_Percentile": values.quantile(0.99),
            "Max_Probability": values.max()
        })

prob_summary = pd.DataFrame(rows)

print("\nProbability distributions by true class:")
display(prob_summary.round(4))

# Focus specifically on true S and F beats
for target_class in ["S", "F"]:
    subset = probs_df[probs_df["True_Label"] == target_class]
    values = subset[f"Prob_{target_class}"]

    print(f"\nActual {target_class} beats:")
    print("Mean probability:", round(values.mean(), 4))
    print("Median probability:", round(values.median(), 4))
    print("Fraction with probability >= 0.10:",
          round((values >= 0.10).mean(), 4))
    print("Fraction with probability >= 0.25:",
          round((values >= 0.25).mean(), 4))
    print("Fraction with probability >= 0.50:",
          round((values >= 0.50).mean(), 4))

Rows: 525235
Unique beats: 105047

Probability distributions by true class:


,Actual_Class,Probability_Target,Count,Mean_Probability,Median_Probability,90th_Percentile,99th_Percentile,Max_Probability
0,N,S,451705,0.0537,0.025,0.145,0.3350,0.785
1,N,F,451705,0.0299,0.005,0.085,0.3750,0.855
2,S,S,13905,0.0446,0.020,0.115,0.3248,0.550
3,S,F,13905,0.0133,0.000,0.025,0.2200,0.850
4,V,S,36145,0.0168,0.000,0.050,0.1650,0.540
5,V,F,36145,0.0321,0.005,0.095,0.3300,0.795
6,F,S,4010,0.0331,0.015,0.090,0.1900,0.415
7,F,F,4010,0.0373,0.020,0.090,0.2745,0.575
8,Q,S,19470,0.0168,0.005,0.040,0.1616,0.460
9,Q,F,19470,0.0226,0.010,0.060,0.1450,0.270



Actual S beats:
Mean probability: 0.0446
Median probability: 0.02
Fraction with probability >= 0.10: 0.1274
Fraction with probability >= 0.25: 0.0234
Fraction with probability >= 0.50: 0.0007

Actual F beats:
Mean probability: 0.0373
Median probability: 0.02
Fraction with probability >= 0.10: 0.0783
Fraction with probability >= 0.25: 0.012
Fraction with probability >= 0.50: 0.001


In [90]:
import numpy as np
import pandas as pd

# Confirm the F4 feature matrix and labels
X_f4 = X_F4
y_array = np.asarray(y)
patient_array = np.asarray(person_ids)

classes = ["N", "S", "V", "F", "Q"]

print("Feature matrix:", X_f4.shape)
print("Labels:", y_array.shape)
print("Patients:", len(np.unique(patient_array)))
print("Class counts:")
print(pd.Series(y_array).value_counts().reindex(classes, fill_value=0))

# Confirm the saved outer-fold setup
print("\nOuter repeats:", len(split_indices))
print("Outer folds:", sum(len(folds) for folds in split_indices.values()))

# Check whether the previously created inner splits are available
try:
    print("Existing inner splits:", len(inner_split_indices))
except NameError:
    print("inner_split_indices is not currently defined.")

Feature matrix: (105047, 12)
Labels: (105047,)
Patients: 45
Class counts:
N    90341
S     2781
V     7229
F      802
Q     3894
Name: count, dtype: int64

Outer repeats: 5
Outer folds: 25
Existing inner splits: 25


In [91]:
import numpy as np
import pandas as pd

classes = ["N", "S", "V", "F", "Q"]
coverage_rows = []

for key, (inner_train_idx, inner_val_idx) in inner_split_indices.items():
    val_labels = y_array[inner_val_idx]
    counts = pd.Series(val_labels).value_counts()

    row = {
        "Split": str(key),
        "Inner_Train_Patients": len(set(patient_array[inner_train_idx])),
        "Inner_Val_Patients": len(set(patient_array[inner_val_idx])),
    }

    for c in classes:
        row[f"Val_{c}_Count"] = int(counts.get(c, 0))

    coverage_rows.append(row)

coverage_df = pd.DataFrame(coverage_rows)

print("Inner validation class counts:")
display(coverage_df)

print("\nNumber of inner splits with zero examples per class:")
display(
    pd.DataFrame({
        "Class": classes,
        "Splits_With_Zero": [
            int((coverage_df[f"Val_{c}_Count"] == 0).sum())
            for c in classes
        ],
        "Splits_With_Fewer_Than_10": [
            int((coverage_df[f"Val_{c}_Count"] < 10).sum())
            for c in classes
        ]
    })
)

Inner validation class counts:


,Split,Inner_Train_Patients,Inner_Val_Patients,Val_N_Count,Val_S_Count,Val_V_Count,Val_F_Count,Val_Q_Count
0,"(1, 1)",28,8,13052,393,1033,2,3885
1,"(1, 2)",29,8,16467,648,1142,7,5
2,"(1, 3)",28,8,19842,245,1225,6,2
3,"(1, 4)",28,7,12287,108,810,380,2079
4,"(1, 5)",28,8,15629,327,1354,5,2082
5,"(2, 1)",28,8,17902,310,1502,18,2
6,"(2, 2)",29,8,12192,1496,1043,22,2077
7,"(2, 3)",28,8,16933,657,2313,397,2
8,"(2, 4)",28,7,15008,33,254,12,5
9,"(2, 5)",28,8,15959,179,579,7,2



Number of inner splits with zero examples per class:


,Class,Splits_With_Zero,Splits_With_Fewer_Than_10
0,N,0,0
1,S,0,0
2,V,0,0
3,F,0,8
4,Q,5,19


In [92]:
import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score

classes = ["N", "S", "V", "F", "Q"]

def make_class_weights(y_train, strategy):
    labels, counts = np.unique(y_train, return_counts=True)
    count_map = dict(zip(labels, counts))
    n = len(y_train)

    if strategy == "no_weights":
        return None

    # Standard inverse-frequency weighting
    raw = {c: n / (len(labels) * count_map[c]) for c in labels}

    if strategy == "balanced":
        return raw

    if strategy == "sqrt_balanced":
        return {c: np.sqrt(w) for c, w in raw.items()}

    if strategy == "capped_balanced":
        reference = raw.get("N", 1.0)
        cap = 5.0 * reference
        return {c: min(w, cap) for c, w in raw.items()}

    raise ValueError(f"Unknown strategy: {strategy}")

strategies = [
    "no_weights",
    "balanced",
    "sqrt_balanced",
    "capped_balanced"
]

inner_results = []

for split_key, (inner_train_idx, inner_val_idx) in inner_split_indices.items():
    X_train = X_f4.iloc[inner_train_idx] if isinstance(X_f4, pd.DataFrame) else X_f4[inner_train_idx]
    X_val = X_f4.iloc[inner_val_idx] if isinstance(X_f4, pd.DataFrame) else X_f4[inner_val_idx]

    y_train = y_array[inner_train_idx]
    y_val = y_array[inner_val_idx]

    for strategy in strategies:
        weights = make_class_weights(y_train, strategy)

        model = RandomForestClassifier(
            n_estimators=100,
            class_weight=weights,
            random_state=42,
            n_jobs=-1
        )
        model.fit(X_train, y_train)
        pred = model.predict(X_val)

        inner_results.append({
            "Split": str(split_key),
            "Strategy": strategy,
            "Macro_F1": f1_score(
                y_val, pred,
                labels=classes,
                average="macro",
                zero_division=0
            )
        })

    print(f"Completed inner split {split_key}")

inner_results_df = pd.DataFrame(inner_results)

inner_summary = (
    inner_results_df
    .groupby("Strategy")["Macro_F1"]
    .agg(["mean", "std", "median"])
    .sort_values("mean", ascending=False)
    .reset_index()
)

print("\nInner-validation Macro-F1 by strategy:")
display(inner_summary.round(4))

inner_results_df.to_csv(
    "/kaggle/working/phase2_F8_inner_weight_results.csv",
    index=False
)

inner_summary.to_csv(
    "/kaggle/working/phase2_F8_inner_weight_summary.csv",
    index=False
)

print("Saved F8 inner-validation results.")

Completed inner split (1, 1)
Completed inner split (1, 2)
Completed inner split (1, 3)
Completed inner split (1, 4)
Completed inner split (1, 5)
Completed inner split (2, 1)
Completed inner split (2, 2)
Completed inner split (2, 3)
Completed inner split (2, 4)
Completed inner split (2, 5)
Completed inner split (3, 1)
Completed inner split (3, 2)
Completed inner split (3, 3)
Completed inner split (3, 4)
Completed inner split (3, 5)
Completed inner split (4, 1)
Completed inner split (4, 2)
Completed inner split (4, 3)
Completed inner split (4, 4)
Completed inner split (4, 5)
Completed inner split (5, 1)
Completed inner split (5, 2)
Completed inner split (5, 3)
Completed inner split (5, 4)
Completed inner split (5, 5)

Inner-validation Macro-F1 by strategy:


,Strategy,mean,std,median
0,balanced,0.3419,0.0467,0.3355
1,sqrt_balanced,0.3393,0.0486,0.3344
2,capped_balanced,0.3391,0.0494,0.3367
3,no_weights,0.3391,0.0561,0.3447


Saved F8 inner-validation results.


In [93]:

import os
import pickle
import shutil
from pathlib import Path
import numpy as np
import pandas as pd

# Create a dedicated checkpoint folder
SAVE_DIR = Path("/kaggle/working/F8_resume")
SAVE_DIR.mkdir(parents=True, exist_ok=True)

# Find the F4 feature matrix and labels used in F8
features = X_f4 if "X_f4" in globals() else X_F4
labels = y_array if "y_array" in globals() else y

# Build a resume checkpoint: no model training is performed here
checkpoint = {
    "stage": "F8_inner_complete_outer_evaluation_pending",
    "features": features,
    "labels": np.asarray(labels),
    "classes": ["N", "S", "V", "F", "Q"],
    "split_indices": split_indices,
    "inner_split_indices": inner_split_indices,
    "inner_results": inner_results_df,
    "inner_summary": inner_summary,
    "selected_strategy": "balanced",
    "random_state": 42,
    "n_estimators": 100,
    "completed_inner_splits": 25,
}

# Include the F4 baseline fold metrics, if the saved file exists
f4_metrics_path = Path(
    "/kaggle/working/phase2_F3_F4_fold_metrics.csv"
)

if f4_metrics_path.exists():
    checkpoint["f4_fold_metrics"] = pd.read_csv(f4_metrics_path)
    shutil.copy2(
        f4_metrics_path,
        SAVE_DIR / "phase2_F3_F4_fold_metrics.csv"
    )

# Save the main checkpoint
checkpoint_path = SAVE_DIR / "F8_resume_checkpoint.pkl"

with open(checkpoint_path, "wb") as f:
    pickle.dump(checkpoint, f, protocol=pickle.HIGHEST_PROTOCOL)

# Save readable copies of the F8 inner-validation results
inner_results_df.to_csv(
    SAVE_DIR / "phase2_F8_inner_weight_results.csv",
    index=False
)
inner_summary.to_csv(
    SAVE_DIR / "phase2_F8_inner_weight_summary.csv",
    index=False
)

# Write a short resume note for tomorrow
resume_note = """
ECG ARRHYTHMIA CLASSIFICATION — RESUME NOTES

Completed:
- F4 feature set: 12 MLII features
- Patient-grouped repeated 5-fold outer CV: 25 folds
- F8 inner-validation comparison: 25 splits, 4 weighting strategies
- F8 inner results and summary saved

F8 inner-validation mean Macro-F1:
- balanced:       0.3419
- sqrt_balanced:  0.3393
- capped_balanced:0.3391
- no_weights:     0.3391

Selected candidate: balanced class weights
Random Forest: 100 trees, random_state=42, n_jobs=-1

NEXT STEP:
Evaluate the selected balanced-weight strategy on the EXISTING
25 outer folds, keeping those folds unchanged. Compare against F4
using mean outer-fold Macro-F1. Then save results and stop at F8.

Do not start F9 or additional experiments.
"""

(SAVE_DIR / "RESUME_NOTES.txt").write_text(
    resume_note.strip() + "\n",
    encoding="utf-8"
)

print("Checkpoint saved successfully.")
print(f"Checkpoint: {checkpoint_path}")
print(f"Checkpoint size: {checkpoint_path.stat().st_size / (1024**2):.2f} MB")
print("\nFiles saved:")
for p in sorted(SAVE_DIR.iterdir()):
    print(f" - {p.name} ({p.stat().st_size / (1024**2):.2f} MB)")


Checkpoint saved successfully.
Checkpoint: /kaggle/working/F8_resume/F8_resume_checkpoint.pkl
Checkpoint size: 41.08 MB

Files saved:
 - F8_resume_checkpoint.pkl (41.08 MB)
 - RESUME_NOTES.txt (0.00 MB)
 - phase2_F3_F4_fold_metrics.csv (0.00 MB)
 - phase2_F8_inner_weight_results.csv (0.00 MB)
 - phase2_F8_inner_weight_summary.csv (0.00 MB)


In [94]:
import pandas as pd

comparison = pd.DataFrame([
    ["F1", 0.879840, 0.334017, 0.322155],
    ["F2", 0.875829, 0.368825, 0.342625],
    ["F3", 0.900351, 0.432132, 0.395877],
    ["F4", 0.909523, 0.457877, 0.411359],
    ["F5", 0.914681, 0.433791, 0.407025],
    ["F6", 0.9021,   0.4592,   0.4101],
    ["F7", 0.9184,   0.4426,   0.4098],
], columns=[
    "Experiment", "Mean_Accuracy",
    "Mean_Balanced_Accuracy", "Mean_Macro_F1"
])

comparison.to_csv(
    "/kaggle/working/F1_F7_comparison.csv",
    index=False
)

print("Saved F1–F7 comparison.")
display(comparison)

Saved F1–F7 comparison.


,Experiment,Mean_Accuracy,Mean_Balanced_Accuracy,Mean_Macro_F1
0,F1,0.879840,0.334017,0.322155
1,F2,0.875829,0.368825,0.342625
2,F3,0.900351,0.432132,0.395877
3,F4,0.909523,0.457877,0.411359
4,F5,0.914681,0.433791,0.407025
5,F6,0.902100,0.459200,0.410100
6,F7,0.918400,0.442600,0.409800
